# CP-1: expanded YOLO11 congestion experiment

Configured for `RUN_MODE = 'expanded'`, `DETECTOR = 'yolo11'`.
Connect a Colab GPU (such as T4), then run from the top. Only edit the configuration
cell; the saved feature-data cell contains both detectors and should stay unchanged.

The notebook scans all four camera folders and matches filenames to
`CP-1 Dataset/CSV/Train.csv`. MP4 files do not need embedded attributes.
It reports accessible, labelled, missing and unlabelled videos before extraction.

Default selection is up to 400 additional training clips plus 48 existing training
clips, and up to 64 new evaluation clips. Selection is by camera/date without
target-class balancing. Historical test clips, their target clips and nearby
source/target times are excluded from the new evaluation. It does not process all
16,000 videos. `PLAN_ONLY = True` saves an inventory and proposed plan without GPU
processing; leave it `False` to run the selected experiment.

Results, models, logs, checkpoints and ZIPs are saved to
`My Drive/CP-1 Dataset/Forecasting_v3/yolo11/expanded_01/`.
Completed clips resume after a disconnect: reconnect the same Drive account and
run from the top with unchanged settings. An interrupted clip restarts from its
first frame. Choose a new `EXPERIMENT_NAME` for a changed selection/configuration.

Model selection uses chronological validation on training dates. Final metrics
include macro-F1, balanced accuracy, class support and same-sample baselines.
Compare these baselines on the expanded sample; its class proportions differ
from the earlier balanced 32-clip test. Motion features require complete saved-log
coverage and represent image-plane movement, not calibrated road speed.

The recording dates were examined previously. This is broader evaluation coverage,
not new independent dates, and higher accuracy is not guaranteed. Current paper
results and completed cached exports are preserved. Any saved outputs below are
from the prior cached execution until you rerun these updated cells.


In [ ]:
!pip install -q "scikit-learn>=1.4" "xgboost>=2.0" pandas numpy matplotlib joblib

In [ ]:
"""Chronological congestion-model selection, cached features and optional expansion.

Target labels are used only for training/scoring. Prediction inputs never include
current/future human congestion labels. Historical test results are exploratory.
"""
from pathlib import Path
import argparse
import hashlib
import io
import importlib.metadata
import json
import re
import zipfile

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

CLASSES = ['free flowing', 'heavy delay', 'light delay', 'moderate delay']
BASE = ['total_unique_vehicles', 'entering_count', 'exiting_count', 'car_count',
        'truck_count', 'bus_count', 'motorcycle_count', 'avg_vehicles_per_frame',
        'max_vehicles_per_frame', 'flow_rate_per_min', 'avg_bbox_area',
        'std_bbox_area', 'avg_confidence', 'soft_count']
SEED = 42


def validate_labels(labels):
    required = ['videos', 'view_label', 'datetimestamp_start', 'datetimestamp_end',
                'congestion_enter_rating']
    missing = set(required)-set(labels.columns)
    if missing:
        raise ValueError('Label CSV is missing columns: '+', '.join(sorted(missing)))
    if labels[required].isna().any().any():
        raise ValueError('Label CSV contains missing video identifiers, timestamps or congestion labels.')
    t = labels.copy()
    t['videos'] = t.videos.astype(str).str.replace('\\', '/', regex=False).str.strip()
    if t.videos.eq('').any():
        raise ValueError('Label CSV contains an empty video path.')
    if not t.congestion_enter_rating.isin(CLASSES).all():
        raise ValueError('Label CSV contains unknown congestion classes.')
    for column in ['datetimestamp_start', 'datetimestamp_end']:
        if pd.to_datetime(t[column], errors='coerce').isna().any():
            raise ValueError('Label CSV contains invalid '+column)
    unique = t[required].drop_duplicates()
    if unique.videos.duplicated().any():
        raise ValueError('The same video has conflicting labels, camera names or timestamps.')
    return t.drop_duplicates('videos').copy()


def label_timing_issues(labels):
    t = validate_labels(labels)
    bad = pd.to_datetime(t.datetimestamp_end) <= pd.to_datetime(t.datetimestamp_start)
    report = t.loc[bad,['videos','datetimestamp_start','datetimestamp_end']].copy()
    report['reason'] = 'segment end is not after segment start'
    return report


def dataset_label_audit(labels, available):
    """Match labels to accessible paths without requiring attributes in MP4 files."""
    labels = validate_labels(labels)
    labelled = set(labels.videos)
    available = set(available)
    invalid_timing = set(label_timing_issues(labels).videos)
    cameras = ['normanniles'+str(i) for i in range(1,5)]
    rows = []
    for camera in cameras:
        known = {key for key in labelled if key.split('/')[0] == camera}
        found = {key for key in available if key.split('/')[0] == camera}
        rows.append(dict(camera_id=camera, labelled_videos=len(known),
                         accessible_videos=len(found), matched_videos=len(known & found),
                         labelled_but_missing=len(known-found), unlabelled_videos=len(found-known),
                         invalid_timing_labels=len(known & invalid_timing)))
    missing = labels[~labels.videos.isin(available)][['videos','view_label','datetimestamp_start']]
    unlabelled = pd.DataFrame({'video_path':sorted(available-labelled)})
    return pd.DataFrame(rows), missing, unlabelled


def prepare_pairs(labels, available):
    t = validate_labels(labels)
    t['source_time'] = pd.to_datetime(t.datetimestamp_start, errors='coerce')
    t['segment_end'] = pd.to_datetime(t.datetimestamp_end, errors='coerce')
    t['segment_duration'] = (t.segment_end-t.source_time).dt.total_seconds()
    t = t.dropna(subset=['source_time', 'segment_end', 'congestion_enter_rating'])
    t = t.sort_values(['view_label', 'source_time', 'videos']).drop_duplicates('videos')
    grouped = t.groupby('view_label', sort=False)
    t['target_time'] = grouped.source_time.shift(-1)
    t['target_video_path'] = grouped.videos.shift(-1)
    t['next_enter_rating'] = grouped.congestion_enter_rating.shift(-1)
    t['target_duration'] = grouped.segment_duration.shift(-1)
    t['gap_sec'] = (t.target_time - t.segment_end).dt.total_seconds()
    t['horizon_sec'] = (t.target_time - t.source_time).dt.total_seconds()
    t = t[t.videos.isin(available) & t.gap_sec.between(0, 2) & (t.horizon_sec > 0)
          & (t.segment_duration > 0) & (t.target_duration > 0)
          & t.next_enter_rating.isin(CLASSES)].copy()
    t['source_date'] = t.source_time.dt.strftime('%Y-%m-%d')
    t['camera_id'] = t.videos.str.split('/').str[0]
    t = t[t.camera_id.isin(['normanniles'+str(i) for i in range(1, 5)])]
    t['split'] = np.where(t.source_time < pd.Timestamp('2025-10-25'), 'train', 'test')
    # Purge a training target that reaches the test period.
    return t[(t.split != 'train') | (t.target_time < pd.Timestamp('2025-10-25'))]


def expansion_plan(labels, available, historical, per_train_day=20, per_test_day=8):
    """Sample by camera/date, never by the future label; exclude old test neighborhoods."""
    if any(isinstance(n, bool) or not isinstance(n, (int, np.integer)) or n <= 0
           for n in [per_train_day, per_test_day]):
        raise ValueError('Per-camera/date clip limits must be positive integers.')
    t = prepare_pairs(labels, available)
    old_keys = set(historical.video_path)
    old_targets = set(t.loc[t.videos.isin(old_keys),'target_video_path'])
    excluded_keys = old_keys | old_targets
    eligible_test = t.split.eq('test') & ~t.videos.isin(excluded_keys) & ~t.target_video_path.isin(excluded_keys)
    for old in historical.itertuples():
        same_camera = t.camera_id.eq(old.camera_id)
        for time in (pd.Timestamp(old.source_time), pd.Timestamp(old.target_time)):
            near = ((t.source_time-time).abs() <= pd.Timedelta(minutes=2)) | (
                (t.target_time-time).abs() <= pd.Timedelta(minutes=2))
            eligible_test &= ~(same_camera & near)
    chunks = []
    for (_, _, split), group in t.groupby(['camera_id', 'source_date', 'split']):
        if split == 'test': group = group.loc[eligible_test.reindex(group.index)]
        else: group = group[~group.videos.isin(old_keys)]
        n = per_test_day if split == 'test' else per_train_day
        if len(group): chunks.append(group.sample(min(n, len(group)), random_state=SEED))
    # Retain completed training clips to save compute, but never old test clips.
    chunks.append(t[t.videos.isin(historical.loc[historical.split == 'train', 'video_path'])])
    result = pd.concat(chunks).drop_duplicates('videos').sort_values(['source_time', 'camera_id'])
    if not (result.split == 'test').any():
        raise ValueError('No new evaluation clips are available. Keep the cached analysis exploratory.')
    return result.reset_index(drop=True)


def contract_signature(configuration):
    return hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()


def plan_fingerprint(plan):
    """Hash the actual source/target assignments, independently of CSV formatting."""
    columns = ['videos','view_label','target_video_path','source_time','segment_end',
               'target_time','next_enter_rating','congestion_enter_rating','split']
    missing = set(columns)-set(plan)
    if missing or plan.videos.duplicated().any():
        raise ValueError('Saved plan is missing required fields or has duplicate videos.')
    normalized = plan[columns].copy().reset_index(drop=True).sort_values('videos')
    for column in ['source_time','segment_end','target_time']:
        normalized[column] = pd.to_datetime(normalized[column],errors='raise').dt.strftime('%Y-%m-%dT%H:%M:%S.%f')
    if normalized.isna().any().any():
        raise ValueError('Saved plan contains missing assignments.')
    payload = json.dumps(normalized.to_dict('records'),sort_keys=True,separators=(',',':'))
    return hashlib.sha256(payload.encode()).hexdigest()


def resume_saved_plan(proposed, labels, available, historical, folder, sampling_limits):
    """Reuse a verified frozen selection, never replace it with a new random draw."""
    folder = Path(folder)
    contract_path, plan_path = folder/'checkpoint_contract.json', folder/'selected_pairs.csv'
    if not contract_path.exists():
        return proposed, None
    saved = json.loads(contract_path.read_text())
    previous = saved['configuration']
    if saved.get('signature') != contract_signature(previous):
        raise RuntimeError('Saved checkpoint contract failed its integrity check. Files were not replaced.')
    if not plan_path.exists():
        raise RuntimeError('Saved contract has no selected_pairs.csv. Restore that file before resuming.')
    if hashlib.sha256(plan_path.read_bytes()).hexdigest() != previous.get('plan_sha256'):
        raise RuntimeError('The saved selected_pairs.csv changed after checkpoint creation. Restore the original plan; files were not replaced.')
    frozen = pd.read_csv(plan_path)
    if previous.get('sampling_limits') is not None:
        if previous['sampling_limits'] != sampling_limits:
            raise RuntimeError('Sampling limits changed. Restore them or use a new EXPERIMENT_NAME; the checkpoint is unchanged.')
    else:
        # v4 did not record requested limits. Require the same realized group sizes.
        keys = ['split','camera_id','source_date']
        old_sizes = frozen.groupby(keys).size().sort_index()
        new_sizes = proposed.groupby(keys).size().sort_index()
        if not old_sizes.equals(new_sizes):
            raise RuntimeError('Sampling counts or accessible videos changed from the saved v4 plan. Restore the original settings and video availability.')
    valid = prepare_pairs(labels,available).set_index('videos',drop=False)
    missing = set(frozen.videos)-set(valid.index)
    if missing:
        raise RuntimeError(f'{len(missing)} saved clips are inaccessible or no longer form valid labelled pairs. Restore the original videos/labels.')
    if plan_fingerprint(frozen) != plan_fingerprint(valid.loc[frozen.videos]):
        raise RuntimeError('Source/target labels or timestamps changed for saved clips. Restore the original label CSV.')
    old_test = set(historical.loc[historical.split.eq('test'),'video_path'])
    test = frozen[frozen.split.eq('test')]
    if (set(test.videos) | set(test.target_video_path)) & old_test:
        raise RuntimeError('The saved evaluation selection overlaps historical test clips.')
    for column in ['source_time','target_time','segment_end']:
        frozen[column] = pd.to_datetime(frozen[column],errors='raise')
    print('Resuming the verified saved clip plan:',len(frozen),'clips.')
    return frozen, saved


def checkpoint_changes(saved, current):
    """Ignore tuple/list JSON representation; allow only verified v4 metadata migration."""
    previous = json.loads(json.dumps(saved['configuration']))
    current = json.loads(json.dumps(current))
    if previous.get('version') == 4 and current.get('version') == 5:
        keys = set(previous)-{'version'}
    else:
        keys = set(previous) | set(current)
    return {key:{'saved':previous.get(key),'current':current.get(key)}
            for key in sorted(keys) if previous.get(key) != current.get(key)}


def motion_features(observations, row):
    """Image-plane motion proxies; not calibrated road speed or true occupancy."""
    frames = int(row['total_frames'])
    fps = float(row['fps'])
    width, height = map(float, row['resolution'].split('x'))
    if frames <= 0 or fps <= 0: raise ValueError('Invalid video timing')
    o = observations.copy()
    if o.empty:
        return dict(motion_speed_median=0., motion_speed_p90=0., slow_observation_fraction=0.,
                    bbox_occupancy_proxy=0., density_trend=0., density_std=0.)
    numeric = ['frame_idx', 'track_id', 'cx', 'cy', 'x1', 'y1', 'x2', 'y2']
    o[numeric] = o[numeric].apply(pd.to_numeric, errors='raise')
    if o[numeric].isna().any().any(): raise ValueError('Missing tracking observations')
    if not o.frame_idx.between(0, frames-1).all(): raise ValueError('Observation frames exceed clip')
    if o.duplicated(['frame_idx', 'track_id']).any(): raise ValueError('Duplicate track observations')
    o = o.sort_values(['track_id', 'frame_idx'])
    dt = o.groupby('track_id').frame_idx.diff() / fps
    dx = o.groupby('track_id').cx.diff()
    dy = o.groupby('track_id').cy.diff()
    speed = (np.hypot(dx, dy) / (dt * np.hypot(width, height)))[dt.between(1/fps-1e-8, .5)]
    density = np.bincount(o.frame_idx.astype(int), minlength=frames)
    third = max(1, frames // 3)
    areas = (o.x2-o.x1).clip(lower=0) * (o.y2-o.y1).clip(lower=0)
    return dict(motion_speed_median=float(speed.median()) if len(speed) else 0.,
                motion_speed_p90=float(speed.quantile(.9)) if len(speed) else 0.,
                slow_observation_fraction=float((speed < .005).mean()) if len(speed) else 0.,
                bbox_occupancy_proxy=float(areas.sum() / (frames*width*height)),
                density_trend=float(density[-third:].mean()-density[:third].mean()),
                density_std=float(density.std()))


def add_motion(table, log_dirs):
    table = table.copy()
    extracted = []
    for _, row in table.iterrows():
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', row.video_path)
        found = next((Path(d)/(safe+'_observations.csv') for d in log_dirs
                      if (Path(d)/(safe+'_observations.csv')).is_file()), None)
        extracted.append(motion_features(pd.read_csv(found), row) if found else {})
    motion = pd.DataFrame(extracted, index=table.index)
    # Only use this feature family when every clip has it, avoiding a run-version
    # missingness signal. Cache-only reruns still work with the original features.
    if motion.empty or motion.isna().any().any():
        print(f'Motion logs available for {sum(bool(r) for r in extracted)}/{len(table)} clips; motion family omitted.')
        return table
    return pd.concat([table, motion], axis=1)


def make_inputs(table, enhanced):
    x = table[BASE].apply(pd.to_numeric, errors='raise').copy()
    if enhanced:
        dimensions = table.resolution.str.split('x', expand=True).astype(float)
        area = dimensions[0]*dimensions[1]
        total = x.total_unique_vehicles.clip(lower=1)
        for name in ['entering_count', 'exiting_count', 'car_count', 'truck_count', 'bus_count', 'motorcycle_count']:
            x[name+'_share'] = x[name]/total
        x['bbox_area_fraction'] = x.avg_bbox_area / area
        x['bbox_std_fraction'] = x.std_bbox_area / area
        x['density_to_crossings'] = x.avg_vehicles_per_frame / total
        x['soft_hard_ratio'] = x.soft_count / total
        x['density_peak_ratio'] = x.max_vehicles_per_frame / x.avg_vehicles_per_frame.clip(lower=.1)
        time = pd.to_datetime(table.source_time)
        hour = time.dt.hour + time.dt.minute/60
        x['hour_sin'], x['hour_cos'] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
        # Camera identity is available at deployment; human congestion is not.
        x['camera_id'] = table.camera_id.astype(str)
        for col in table.columns:
            if col in ['motion_speed_median', 'motion_speed_p90', 'slow_observation_fraction',
                       'bbox_occupancy_proxy', 'density_trend', 'density_std']:
                x[col] = pd.to_numeric(table[col], errors='raise')
    return x.replace([np.inf, -np.inf], np.nan)


def candidates():
    # Fixed before looking at the audit set. The original model is a candidate,
    # so validation can reject the proposed changes rather than forcing them.
    return [('original_xgb', False), ('regularized_xgb', False),
            ('regularized_xgb', True), ('shallow_xgb', True),
            ('random_forest', True), ('logistic', True)]


def model_pipeline(name, x):
    cats = [c for c in x if not pd.api.types.is_numeric_dtype(x[c])]
    nums = [c for c in x if c not in cats]
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([('fill', SimpleImputer(strategy='median', keep_empty_features=True)),
                              ('scale', StandardScaler())]), nums),
        ('camera', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cats),
    ])
    shared = dict(objective='multi:softprob', eval_metric='mlogloss', tree_method='hist',
                  random_state=SEED, n_jobs=2)
    if name == 'original_xgb':
        model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=.08,
                              subsample=.9, colsample_bytree=.9, **shared)
    elif name == 'regularized_xgb':
        model = XGBClassifier(n_estimators=150, max_depth=2, learning_rate=.04,
                              min_child_weight=3, reg_lambda=10, reg_alpha=.5,
                              subsample=.85, colsample_bytree=.85, **shared)
    elif name == 'shallow_xgb':
        model = XGBClassifier(n_estimators=100, max_depth=1, learning_rate=.05,
                              min_child_weight=2, reg_lambda=10, **shared)
    elif name == 'random_forest':
        model = RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_leaf=3,
                                        max_features=.7, n_jobs=1, random_state=SEED)
    elif name == 'logistic':
        model = LogisticRegression(C=.3, max_iter=1500, random_state=SEED)
    else: raise ValueError(name)
    return Pipeline([('preprocess', preprocess), ('model', model)])


def fit_model(model, x, y, name):
    weights = np.ones(len(y)) if name == 'original_xgb' else compute_sample_weight('balanced', y)
    return model.fit(x, y, model__sample_weight=weights)


def evaluate_pipeline(table, output, evaluation_kind='reused_historical_holdout'):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    t = table.sort_values(['source_time', 'video_path']).reset_index(drop=True)
    if t.video_path.duplicated().any(): raise ValueError('Duplicate clips')
    if t.model_name.nunique() != 1: raise ValueError('Do not mix detectors in one feature table')
    if not t.next_enter_rating.isin(CLASSES).all(): raise ValueError('Unknown target class')
    t['source_time'] = pd.to_datetime(t.source_time)
    t['target_time'] = pd.to_datetime(t.target_time)
    t['source_date'] = t.source_time.dt.strftime('%Y-%m-%d')
    t['split'] = np.where(t.source_time < pd.Timestamp('2025-10-25'), 'train', 'test')
    if not (t.target_time > t.source_time).all(): raise ValueError('Target must be in the future')
    train = t[t.split == 'train'].copy()
    test = t[t.split == 'test'].copy()
    if len(train) < 20 or len(test) < 8: raise ValueError('Insufficient training or evaluation clips')
    if train.target_time.max() >= test.source_time.min(): raise ValueError('Train target overlaps evaluation period')
    y = train.next_enter_rating.map({c:i for i,c in enumerate(CLASSES)}).to_numpy()
    distribution = pd.crosstab(t['split'], t.next_enter_rating).reindex(
        index=['train','test'], columns=CLASSES, fill_value=0)
    distribution.to_csv(output/'split_class_distribution.csv')
    folds = []
    for day in ['2025-10-22', '2025-10-23', '2025-10-24']:
        boundary = pd.Timestamp(day)
        fit = np.flatnonzero((train.source_time < boundary) & (train.target_time < boundary))
        valid = np.flatnonzero(train.source_date == day)
        if len(fit) >= 12 and len(valid) >= 4 and set(y[fit]) == set(range(4)):
            folds.append((day, fit, valid))
    if len(folds) < 2: raise ValueError('Need two chronological validation folds with all training classes')
    logs, configs = [], {}
    for name, enhanced in candidates():
        key = name + ('_context' if enhanced else '_base')
        x = make_inputs(train, enhanced)
        configs[key] = (name, enhanced)
        for day, fit, valid in folds:
            model = fit_model(model_pipeline(name, x), x.iloc[fit], y[fit], name)
            pred = model.predict(x.iloc[valid])
            logs.append(dict(candidate=key, validation_date=day, train_n=len(fit), validation_n=len(valid),
                             accuracy=accuracy_score(y[valid], pred),
                             macro_f1=f1_score(y[valid], pred, labels=range(4), average='macro', zero_division=0)))
    folds_table = pd.DataFrame(logs)
    folds_table.to_csv(output/'validation_folds.csv', index=False)
    ranking = folds_table.groupby('candidate')[['macro_f1', 'accuracy']].mean().reset_index().sort_values(
        ['macro_f1', 'accuracy', 'candidate'], ascending=[False, False, True])
    ranking.to_csv(output/'validation_ranking.csv', index=False)
    chosen = ranking.iloc[0].candidate
    name, enhanced = configs[chosen]
    x = make_inputs(train, enhanced)
    trained = fit_model(model_pipeline(name, x), x, y, name)
    selection = {'selected_candidate': chosen, 'selection_metric': 'mean chronological validation macro-F1',
                 'detector_model': t.model_name.iloc[0],
                 'feature_columns': list(x), 'train_n': len(train), 'test_n': len(test),
                 'seed': SEED, 'evaluation_kind': evaluation_kind,
                 'packages': {p: importlib.metadata.version(p) for p in ['scikit-learn','xgboost','pandas','numpy']},
                 'input_sha256': hashlib.sha256(t.to_csv(index=False).encode()).hexdigest(),
                 'fresh_independent_dates': False,
                 'note': 'The dates 25-26 October were previously examined. New clips improve evaluation coverage, but are not new independent dates.'}
    # Save the frozen selection before any evaluation predictions or metrics.
    (output/'selected_model.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
    joblib.dump(trained, output/'selected_model.joblib')
    joblib.dump({'classes': CLASSES, 'enhanced': enhanced, 'features': list(x)}, output/'feature_schema.joblib')
    truth = test.next_enter_rating.map({c:i for i,c in enumerate(CLASSES)}).to_numpy()
    pred = trained.predict(make_inputs(test, enhanced))
    original = fit_model(model_pipeline('original_xgb', make_inputs(train, False)), make_inputs(train, False), y, 'original_xgb')
    guesses = {'selected_model': pred, 'original_settings_recomputed': original.predict(make_inputs(test, False)),
               'majority': np.full(len(test), np.bincount(y).argmax()),
               'persistence_label_baseline': test.congestion_enter_rating.map({c:i for i,c in enumerate(CLASSES)}).to_numpy()}
    metrics = []
    for label, guess in guesses.items():
        metrics.append(dict(method=label, n=len(test), correct=int((truth == guess).sum()),
                            accuracy=accuracy_score(truth, guess),
                            balanced_accuracy=balanced_accuracy_score(truth, guess),
                            macro_f1=f1_score(truth, guess, labels=range(4), average='macro', zero_division=0)))
    metrics = pd.DataFrame(metrics)
    metrics.to_csv(output/'evaluation_metrics.csv', index=False)
    result = test[['video_path','source_time','target_time','next_enter_rating']].copy()
    result['prediction'] = [CLASSES[int(i)] for i in pred]
    for method, guess in guesses.items():
        result[method+'_prediction'] = [CLASSES[int(i)] for i in guess]
    result.to_csv(output/'evaluation_predictions.csv', index=False)
    pd.DataFrame(confusion_matrix(truth, pred, labels=range(4)), index=CLASSES, columns=CLASSES).to_csv(output/'confusion_matrix.csv')
    pd.DataFrame(classification_report(truth, pred, labels=range(4), target_names=CLASSES,
                                       output_dict=True, zero_division=0)).T.to_csv(output/'classification_report.csv')
    t.to_csv(output/'features_used.csv', index=False)
    report = ('# Forecasting v3 experiment\n\nSelected using training-date validation only: '+chosen+
              '\n\nEvaluation type: '+evaluation_kind+'\n\n'+metrics.to_string(index=False)+
              '\n\nThese are congestion-classification metrics. Detection/counting accuracy remains unmeasured. '
              'The original-settings baseline is refitted under the current software, sorted row order and preprocessing; '
              'it is not a bit-for-bit replay of the archived 37.5% run. '
              'Historical test performance is exploratory after previous inspection. Do not claim independent confirmation '
              'without new dates/cameras. Context features contain no human congestion labels. Motion is an image-plane proxy.\n')
    (output/'REPORT.md').write_text(report, encoding='utf-8')
    print(ranking.to_string(index=False))
    print('\nSelected:', chosen)
    print(metrics.to_string(index=False))
    return selection, metrics


def read_cached(root, model):
    root = Path(root)
    if model == 'yolov8':
        return pd.read_csv(root/'CP-1/Journal_Results_Export_yolov8_v2/next_segment_features.csv')
    with zipfile.ZipFile(root/'CP-1/Journal_Results_Export_v2-20261020T090346Z-1-001.zip') as archive:
        return pd.read_csv(io.BytesIO(archive.read('Journal_Results_Export_v2/next_segment_features.csv')))

In [ ]:
# Change only this cell to configure the experiment.
RUN_MODE = 'expanded'  # 'expanded': larger GPU run; 'cached': the original 80 clips.
DETECTOR = 'yolo11'  # 'yolo11' or 'yolov8'
DRIVE_BASE = '/content/drive/MyDrive/CP-1 Dataset'
LABELS_RELATIVE_PATH = 'CSV/Train.csv'
EXPERIMENT_NAME = 'expanded_01'  # Use the same name/settings to resume.
TRAIN_CLIPS_PER_CAMERA_DATE = 20
TEST_CLIPS_PER_CAMERA_DATE = 8
PLAN_ONLY = False  # True: audit filenames/labels and save a plan without GPU extraction.
USE_MOTION_FEATURES = True  # Used only when logs cover every selected clip.
assert RUN_MODE in {'cached','expanded'} and DETECTOR in {'yolo11','yolov8'}
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', EXPERIMENT_NAME), 'Use a simple experiment folder name.'

In [ ]:
# Saved feature data: do not edit. Change DETECTOR in the configuration cell.
CACHED_FEATURES = {
    'yolo11': (
        'video_path,filename,camera_id,model_name,total_frames,fps,duration_sec,resolution,counting_line,total_unique_vehicles,entering_count,exiting_count,soft_count,soft_entering_count,soft_exiting_count,avg_vehicles_per_frame,max_vehicles_per_frame,flow_rate_per_min,avg_bbox_area,std_bbox_area,avg_confidence,crossing_log,car_count,motorcycle_count,bus_count,truck_count,videos,source_time,target_time,source_date,split,next_enter_rating,congestion_enter_rating,horizon_sec\r\n'
        'normanniles1/normanniles1_2025-10-25-13-21-45.mp4,normanniles1_2025-10-25-13-21-45.mp4,normanniles1,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",41,25,16,29.7198,17.8716,11.8481,8.565580618212197,16,41.10275689223057,2729.857017458305,5128.344523601557,0.7034023225334352,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-25-13-21-45.mp4_crossings.csv,33,0,0,8,normanniles1/normanniles1_2025-10-25-13-21-45.mp4,2025-10-25 13:21:44,2025-10-25 13:22:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-14-09-45.mp4,normanniles1_2025-10-25-14-09-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",54,25,29,38.9616,17.4115,21.5502,12.161936560934892,17,54.090150250417366,1906.9229238160603,2060.392777529778,0.6951216126434817,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-25-14-09-45.mp4_crossings.csv,46,0,0,8,normanniles1/normanniles1_2025-10-25-14-09-45.mp4,2025-10-25 14:09:44,2025-10-25 14:10:44,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-03-45.mp4,normanniles1_2025-10-25-17-03-45.mp4,normanniles1,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",22,19,3,15.4425,13.1434,2.2992,5.4110275689223055,11,22.05513784461153,2207.47676393392,2943.8882357148977,0.6197254102189949,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-25-17-03-45.mp4_crossings.csv,17,0,0,5,normanniles1/normanniles1_2025-10-25-17-03-45.mp4,2025-10-25 17:03:44,2025-10-25 17:04:44,2025-10-25,test,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-46-45.mp4,normanniles1_2025-10-25-17-46-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",38,20,18,27.0633,13.6017,13.4616,11.030884808013356,14,38.06343906510852,2016.5384033295493,2264.419842214141,0.6767714935300976,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-25-17-46-45.mp4_crossings.csv,30,0,1,7,normanniles1/normanniles1_2025-10-25-17-46-45.mp4,2025-10-25 17:46:44,2025-10-25 17:47:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-55-45.mp4,normanniles1_2025-10-25-17-55-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",42,18,24,30.7612,12.6134,18.1477,14.348080133555928,21,42.07011686143573,1662.5811856419805,2304.5564446410235,0.6733101862325872,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-25-17-55-45.mp4_crossings.csv,37,1,0,4,normanniles1/normanniles1_2025-10-25-17-55-45.mp4,2025-10-25 17:55:44,2025-10-25 17:56:44,2025-10-25,test,heavy delay,light delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-26-13-14-45.mp4,normanniles1_2025-10-26-13-14-45.mp4,normanniles1,yolo11l.pt,1163,20.0,58.15,960x540,"(48, 270)->(912, 270)",31,13,18,21.8712,9.4537,12.4175,6.182287188306105,11,31.986242476354256,1621.2438108484005,2434.369573015769,0.6591098226446228,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-26-13-14-45.mp4_crossings.csv,22,0,0,9,normanniles1/normanniles1_2025-10-26-13-14-45.mp4,2025-10-26 13:14:33,2025-10-26 13:15:42,2025-10-26,test,moderate delay,free flowing,69.0\r\n'
        'normanniles1/normanniles1_2025-10-26-17-16-45.mp4,normanniles1_2025-10-26-17-16-45.mp4,normanniles1,yolo11l.pt,1161,20.0,58.05,960x540,"(48, 270)->(912, 270)",29,19,10,20.197,13.4562,6.7408,5.347975882859604,11,29.97416020671835,1159.1918183282332,1724.100153258924,0.6469834152477714,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-26-17-16-45.mp4_crossings.csv,26,0,0,3,normanniles1/normanniles1_2025-10-26-17-16-45.mp4,2025-10-26 17:16:39,2025-10-26 17:17:42,2025-10-26,test,moderate delay,free flowing,63.0\r\n'
        'normanniles1/normanniles1_2025-10-26-17-39-45.mp4,normanniles1_2025-10-26-17-39-45.mp4,normanniles1,yolo11l.pt,1160,20.0,58.0,960x540,"(48, 270)->(912, 270)",38,21,17,26.7289,14.7044,12.0245,9.841379310344829,18,39.310344827586206,1833.2577084793272,2351.43623677053,0.6389372032984162,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-26-17-39-45.mp4_crossings.csv,28,0,1,9,normanniles1/normanniles1_2025-10-26-17-39-45.mp4,2025-10-26 17:39:33,2025-10-26 17:40:42,2025-10-26,test,free flowing,light delay,69.0\r\n'
        'normanniles2/normanniles2_2025-10-25-12-44-45.mp4,normanniles2_2025-10-25-12-44-45.mp4,normanniles2,yolo11l.pt,1074,20.0,53.7,960x540,"(48, 270)->(912, 270)",19,13,6,15.0719,10.3103,4.7617,5.852886405959032,9,21.22905027932961,3275.0361119949093,2792.341850488737,0.6644520250736075,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-25-12-44-45.mp4_crossings.csv,15,0,1,3,normanniles2/normanniles2_2025-10-25-12-44-45.mp4,2025-10-25 12:44:47,2025-10-25 12:45:45,2025-10-25,test,light delay,light delay,58.0\r\n'
        'normanniles2/normanniles2_2025-10-25-13-08-45.mp4,normanniles2_2025-10-25-13-08-45.mp4,normanniles2,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",24,10,14,18.1434,7.9239,10.2195,8.116666666666667,13,24.0,2932.27340862423,3165.347534872091,0.6950313729010293,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-25-13-08-45.mp4_crossings.csv,17,0,2,5,normanniles2/normanniles2_2025-10-25-13-08-45.mp4,2025-10-25 13:08:47,2025-10-25 13:09:47,2025-10-25,test,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-14-23-45.mp4,normanniles2_2025-10-25-14-23-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",23,14,9,18.0284,11.2488,6.7796,12.718697829716191,16,23.03839732888147,2270.5503051781848,2129.056376871276,0.6553541886102945,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-25-14-23-45.mp4_crossings.csv,20,0,0,3,normanniles2/normanniles2_2025-10-25-14-23-45.mp4,2025-10-25 14:23:47,2025-10-25 14:24:47,2025-10-25,test,moderate delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-16-49-45.mp4,normanniles2_2025-10-25-16-49-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",22,11,11,16.6501,8.8935,7.7567,9.934891485809684,18,22.036727879799667,2576.0374726936648,2371.4532302430744,0.690989591100212,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-25-16-49-45.mp4_crossings.csv,16,0,1,5,normanniles2/normanniles2_2025-10-25-16-49-45.mp4,2025-10-25 16:49:48,2025-10-25 16:50:48,2025-10-25,test,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-17-44-45.mp4,normanniles2_2025-10-25-17-44-45.mp4,normanniles2,yolo11l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",22,10,12,17.1419,7.9029,9.239,8.700668896321071,14,22.073578595317727,2515.227657120892,2545.0570486539887,0.712675758632655,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-25-17-44-45.mp4_crossings.csv,15,1,0,6,normanniles2/normanniles2_2025-10-25-17-44-45.mp4,2025-10-25 17:44:48,2025-10-25 17:45:48,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-06-29-45.mp4,normanniles2_2025-10-26-06-29-45.mp4,normanniles2,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",8,3,5,6.2201,2.4968,3.7233,1.862155388471178,5,8.020050125313283,3150.8748317631225,1805.921620666345,0.6201293285737235,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-26-06-29-45.mp4_crossings.csv,8,0,0,0,normanniles2/normanniles2_2025-10-26-06-29-45.mp4,2025-10-26 06:29:46,2025-10-26 06:30:46,2025-10-26,test,free flowing,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-14-05-45.mp4,normanniles2_2025-10-26-14-05-45.mp4,normanniles2,yolo11l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",15,6,9,11.2706,4.5127,6.7579,5.165551839464883,8,15.050167224080267,2538.8405632890904,1997.285627694292,0.6886187306161778,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-26-14-05-45.mp4_crossings.csv,10,0,0,5,normanniles2/normanniles2_2025-10-26-14-05-45.mp4,2025-10-26 14:05:47,2025-10-26 14:06:47,2025-10-26,test,light delay,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-15-02-45.mp4,normanniles2_2025-10-26-15-02-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",20,11,9,15.2133,8.8327,6.3806,6.487479131886477,10,20.03338898163606,2587.5176273803395,2283.848502037856,0.6589676906823023,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-26-15-02-45.mp4_crossings.csv,17,0,0,3,normanniles2/normanniles2_2025-10-26-15-02-45.mp4,2025-10-26 15:02:47,2025-10-26 15:03:47,2025-10-26,test,free flowing,moderate delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-09-33-45.mp4,normanniles3_2025-10-25-09-33-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",10,4,6,7.5705,3.2638,4.3067,0.7420701168614358,4,10.01669449081803,3998.901012373453,1981.728586468496,0.7564093187468124,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-25-09-33-45.mp4_crossings.csv,9,0,0,1,normanniles3/normanniles3_2025-10-25-09-33-45.mp4,2025-10-25 09:33:44,2025-10-25 09:34:44,2025-10-25,test,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-11-07-45.mp4,normanniles3_2025-10-25-11-07-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",33,9,24,23.4296,7.2433,16.1863,5.08347245409015,10,33.0550918196995,2136.2942528735634,2180.526977880388,0.6769028793567752,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-25-11-07-45.mp4_crossings.csv,23,0,0,10,normanniles3/normanniles3_2025-10-25-11-07-45.mp4,2025-10-25 11:07:44,2025-10-25 11:08:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-13-00-45.mp4,normanniles3_2025-10-25-13-00-45.mp4,normanniles3,yolo11l.pt,770,20.0,38.5,960x540,"(48, 270)->(912, 270)",22,4,18,16.3735,3.4956,12.8779,13.209090909090907,20,34.285714285714285,1507.1061842493364,1729.2511722949314,0.5955448366152937,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-25-13-00-45.mp4_crossings.csv,20,0,0,2,normanniles3/normanniles3_2025-10-25-13-00-45.mp4,2025-10-25 13:00:39,2025-10-25 13:01:39,2025-10-25,test,heavy delay,heavy delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-16-14-45.mp4,normanniles3_2025-10-25-16-14-45.mp4,normanniles3,yolo11l.pt,1202,20.0,60.1,960x540,"(48, 270)->(912, 270)",35,11,24,26.1309,9.0072,17.1238,13.163893510815308,19,34.94176372712146,1203.5091954749416,1860.41975898086,0.6329594451788365,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-25-16-14-45.mp4_crossings.csv,27,1,0,7,normanniles3/normanniles3_2025-10-25-16-14-45.mp4,2025-10-25 16:14:43,2025-10-25 16:15:44,2025-10-25,test,moderate delay,moderate delay,61.0\r\n'
        'normanniles3/normanniles3_2025-10-25-17-20-45.mp4,normanniles3_2025-10-25-17-20-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",45,16,29,33.8085,12.2781,21.5305,13.657762938230384,18,45.075125208681136,1648.5129568512407,2144.3137117898414,0.6596860496056115,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-25-17-20-45.mp4_crossings.csv,36,0,1,8,normanniles3/normanniles3_2025-10-25-17-20-45.mp4,2025-10-25 17:20:44,2025-10-25 17:21:44,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-13-15-45.mp4,normanniles3_2025-10-26-13-15-45.mp4,normanniles3,yolo11l.pt,1199,20.0,59.95,960x540,"(48, 270)->(912, 270)",26,8,18,18.7819,6.0245,12.7574,7.725604670558799,14,26.021684737281067,1113.2912663284033,1555.587396564307,0.6577962366344282,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-26-13-15-45.mp4_crossings.csv,23,0,0,3,normanniles3/normanniles3_2025-10-26-13-15-45.mp4,2025-10-26 13:15:43,2025-10-26 13:16:43,2025-10-26,test,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-15-40-45.mp4,normanniles3_2025-10-26-15-40-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",35,8,27,25.5138,5.539,19.9749,10.447412353923204,16,35.058430717863104,1307.2296260786193,1662.0315221454728,0.6449383215546565,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-26-15-40-45.mp4_crossings.csv,26,0,0,9,normanniles3/normanniles3_2025-10-26-15-40-45.mp4,2025-10-26 15:40:43,2025-10-26 15:41:43,2025-10-26,test,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-17-30-45.mp4,normanniles3_2025-10-26-17-30-45.mp4,normanniles3,yolo11l.pt,1164,20.0,58.2,960x540,"(48, 270)->(912, 270)",28,11,17,20.7718,8.9458,11.8259,8.420103092783505,16,28.865979381443296,1697.6715641261096,2438.2953364422533,0.6354673801609727,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-26-17-30-45.mp4_crossings.csv,23,0,0,5,normanniles3/normanniles3_2025-10-26-17-30-45.mp4,2025-10-26 17:30:39,2025-10-26 17:31:41,2025-10-26,test,free flowing,light delay,62.0\r\n'
        'normanniles4/normanniles4_2025-10-25-12-05-45.mp4,normanniles4_2025-10-25-12-05-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",22,10,12,14.9588,6.6265,8.3322,6.1375,11,22.0,2261.0412763068566,2798.5370367186883,0.6944241911845594,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-25-12-05-45.mp4_crossings.csv,14,0,2,6,normanniles4/normanniles4_2025-10-25-12-05-45.mp4,2025-10-25 12:05:44,2025-10-25 12:06:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-13-51-45.mp4,normanniles4_2025-10-25-13-51-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",25,20,5,18.1377,14.1822,3.9555,12.655833333333334,19,25.0,2009.3402251925988,2505.1267523563483,0.6945773634232707,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-25-13-51-45.mp4_crossings.csv,18,0,2,5,normanniles4/normanniles4_2025-10-25-13-51-45.mp4,2025-10-25 13:51:44,2025-10-25 13:52:44,2025-10-25,test,heavy delay,moderate delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-14-05-45.mp4,normanniles4_2025-10-25-14-05-45.mp4,normanniles4,yolo11l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",26,13,13,17.8778,8.2122,9.6655,13.621981681931723,19,25.978351373855123,1512.1507334963326,1924.0913141599133,0.6324118530209242,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-25-14-05-45.mp4_crossings.csv,20,0,0,6,normanniles4/normanniles4_2025-10-25-14-05-45.mp4,2025-10-25 14:05:44,2025-10-25 14:06:44,2025-10-25,test,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-14-28-45.mp4,normanniles4_2025-10-25-14-28-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",30,13,17,20.3989,8.7476,11.6513,12.959166666666668,17,30.0,1651.1578676612437,1936.9737725050168,0.6717018331604371,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-25-14-28-45.mp4_crossings.csv,23,0,0,7,normanniles4/normanniles4_2025-10-25-14-28-45.mp4,2025-10-25 14:28:44,2025-10-25 14:29:44,2025-10-25,test,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-17-05-45.mp4,normanniles4_2025-10-25-17-05-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",20,10,10,13.9509,6.4381,7.5128,10.408333333333331,15,20.0,1856.1829463570857,2035.504093675496,0.6986324785562088,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-25-17-05-45.mp4_crossings.csv,15,0,2,3,normanniles4/normanniles4_2025-10-25-17-05-45.mp4,2025-10-25 17:05:43,2025-10-25 17:06:43,2025-10-25,test,moderate delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-26-06-23-45.mp4,normanniles4_2025-10-26-06-23-45.mp4,normanniles4,yolo11l.pt,1244,20.0,62.2,960x540,"(48, 270)->(912, 270)",8,5,3,5.9843,3.6239,2.3604,1.1109324758842445,4,7.717041800643086,1962.7908827785816,1894.752126598168,0.6740713264205521,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-26-06-23-45.mp4_crossings.csv,7,0,0,1,normanniles4/normanniles4_2025-10-26-06-23-45.mp4,2025-10-26 06:23:42,2025-10-26 06:24:44,2025-10-26,test,free flowing,free flowing,62.0\r\n'
        'normanniles4/normanniles4_2025-10-26-15-12-45.mp4,normanniles4_2025-10-26-15-12-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",34,20,14,22.5287,12.8613,9.6674,11.524166666666666,17,34.0,1288.962470171379,1761.4683538735594,0.6827484178253451,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-26-15-12-45.mp4_crossings.csv,29,0,1,4,normanniles4/normanniles4_2025-10-26-15-12-45.mp4,2025-10-26 15:12:44,2025-10-26 15:13:44,2025-10-26,test,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-26-16-15-45.mp4,normanniles4_2025-10-26-16-15-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",24,17,7,16.395,11.3181,5.0769,10.58,16,24.0,1678.849086326402,1746.0482328794308,0.6909112533018645,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-26-16-15-45.mp4_crossings.csv,21,0,0,3,normanniles4/normanniles4_2025-10-26-16-15-45.mp4,2025-10-26 16:15:44,2025-10-26 16:16:44,2025-10-26,test,light delay,light delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-08-37-45.mp4,normanniles1_2025-10-20-08-37-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",8,2,6,5.2632,1.077,4.1862,0.9006677796327212,4,8.013355592654424,3326.0583873957366,2306.9607839504865,0.5289197786157886,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-20-08-37-45.mp4_crossings.csv,5,0,1,2,normanniles1/normanniles1_2025-10-20-08-37-45.mp4,2025-10-20 08:37:45,2025-10-20 08:38:45,2025-10-20,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-11-18-45.mp4,normanniles1_2025-10-20-11-18-45.mp4,normanniles1,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",43,22,21,31.0041,15.8856,15.1185,17.664160401002505,24,43.1077694235589,1709.0865493757094,2058.6213391011065,0.677604351394975,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-20-11-18-45.mp4_crossings.csv,33,2,0,8,normanniles1/normanniles1_2025-10-20-11-18-45.mp4,2025-10-20 11:18:45,2025-10-20 11:19:45,2025-10-20,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-11-33-45.mp4,normanniles1_2025-10-20-11-33-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",50,22,28,38.3344,16.3345,21.9999,15.766666666666667,21,50.0,1502.2148520084563,1810.5404068342184,0.7232308597589427,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-20-11-33-45.mp4_crossings.csv,46,0,1,3,normanniles1/normanniles1_2025-10-20-11-33-45.mp4,2025-10-20 11:33:44,2025-10-20 11:34:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-14-02-45.mp4,normanniles1_2025-10-20-14-02-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",32,15,17,23.0076,10.598,12.4096,8.11,14,32.0,2100.444615700781,2669.0233881709933,0.6918721820058993,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-20-14-02-45.mp4_crossings.csv,26,0,0,6,normanniles1/normanniles1_2025-10-20-14-02-45.mp4,2025-10-20 14:02:44,2025-10-20 14:03:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-21-08-05-45.mp4,normanniles1_2025-10-21-08-05-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",1,1,0,0.7996,0.7996,0.0,0.0717863105175292,1,1.001669449081803,3970.988372093024,1646.1282940068515,0.7995612288630286,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-21-08-05-45.mp4_crossings.csv,1,0,0,0,normanniles1/normanniles1_2025-10-21-08-05-45.mp4,2025-10-21 08:05:45,2025-10-21 08:06:45,2025-10-21,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-21-10-59-45.mp4,normanniles1_2025-10-21-10-59-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",51,26,25,35.3411,16.8126,18.5285,15.4475,19,51.0,1495.3174731617846,1718.95164237069,0.6889241590351147,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-21-10-59-45.mp4_crossings.csv,47,0,0,4,normanniles1/normanniles1_2025-10-21-10-59-45.mp4,2025-10-21 10:59:44,2025-10-21 11:00:45,2025-10-21,train,moderate delay,heavy delay,61.0\r\n'
        'normanniles1/normanniles1_2025-10-21-17-31-45.mp4,normanniles1_2025-10-21-17-31-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",43,17,26,30.5193,11.6283,18.8909,14.7,21,43.0,1833.6892857142857,2135.8222098066462,0.6583516620141782,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-21-17-31-45.mp4_crossings.csv,31,0,1,11,normanniles1/normanniles1_2025-10-21-17-31-45.mp4,2025-10-21 17:31:44,2025-10-21 17:32:44,2025-10-21,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-09-03-45.mp4,normanniles1_2025-10-23-09-03-45.mp4,normanniles1,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",4,1,3,2.8821,0.7506,2.1316,0.5951585976627712,3,4.006677796327212,2817.1977559607294,2758.2805204213328,0.7090936329471781,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-23-09-03-45.mp4_crossings.csv,3,0,0,1,normanniles1/normanniles1_2025-10-23-09-03-45.mp4,2025-10-23 09:03:44,2025-10-23 09:04:44,2025-10-23,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-10-24-45.mp4,normanniles1_2025-10-23-10-24-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",31,9,22,23.3718,6.5811,16.7907,10.960833333333332,18,31.0,1938.0093514787504,2449.1955388121773,0.6587629725790237,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-23-10-24-45.mp4_crossings.csv,22,0,0,9,normanniles1/normanniles1_2025-10-23-10-24-45.mp4,2025-10-23 10:24:44,2025-10-23 10:25:44,2025-10-23,train,moderate delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-12-35-45.mp4,normanniles1_2025-10-23-12-35-45.mp4,normanniles1,yolo11l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",44,28,16,30.9777,19.0867,11.891,9.544314381270905,13,44.147157190635454,1786.7676741130092,2236.7603191950047,0.7098143964464133,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-23-12-35-45.mp4_crossings.csv,41,0,0,3,normanniles1/normanniles1_2025-10-23-12-35-45.mp4,2025-10-23 12:35:44,2025-10-23 12:36:44,2025-10-23,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-17-50-45.mp4,normanniles1_2025-10-23-17-50-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",41,18,23,29.6634,12.4636,17.1998,12.130833333333332,19,41.0,1828.043621625335,3190.536811540093,0.6633507794664544,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-23-17-50-45.mp4_crossings.csv,31,1,0,9,normanniles1/normanniles1_2025-10-23-17-50-45.mp4,2025-10-23 17:50:44,2025-10-23 17:51:44,2025-10-23,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-24-15-54-45.mp4,normanniles1_2025-10-24-15-54-45.mp4,normanniles1,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",48,22,26,33.2695,14.455,18.8145,14.415833333333332,19,48.0,2044.341522631366,2343.110053602953,0.6629253442686853,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles1_normanniles1_2025-10-24-15-54-45.mp4_crossings.csv,28,1,1,18,normanniles1/normanniles1_2025-10-24-15-54-45.mp4,2025-10-24 15:54:43,2025-10-24 15:55:44,2025-10-24,train,moderate delay,heavy delay,61.0\r\n'
        'normanniles2/normanniles2_2025-10-20-17-13-45.mp4,normanniles2_2025-10-20-17-13-45.mp4,normanniles2,yolo11l.pt,1191,20.0,59.55,960x540,"(48, 270)->(912, 270)",17,8,9,12.7182,5.8585,6.8597,4.903442485306465,8,17.12846347607053,2407.7666095890413,1811.417886841312,0.7114315519224189,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-20-17-13-45.mp4_crossings.csv,15,0,0,2,normanniles2/normanniles2_2025-10-20-17-13-45.mp4,2025-10-20 17:13:47,2025-10-20 17:14:46,2025-10-20,train,light delay,free flowing,59.0\r\n'
        'normanniles2/normanniles2_2025-10-21-07-06-45.mp4,normanniles2_2025-10-21-07-06-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",1,1,0,0.763,0.763,0.0,1.0968280467445743,3,1.001669449081803,2594.652207001522,1210.449904426146,0.5905412595656727,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-21-07-06-45.mp4_crossings.csv,1,0,0,0,normanniles2/normanniles2_2025-10-21-07-06-45.mp4,2025-10-21 07:06:46,2025-10-21 07:07:46,2025-10-21,train,free flowing,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-21-11-34-45.mp4,normanniles2_2025-10-21-11-34-45.mp4,normanniles2,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",31,17,14,23.7504,13.0574,10.693,9.73266499582289,14,31.07769423558897,2215.1985407725324,2345.6556219174768,0.7229545209902167,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-21-11-34-45.mp4_crossings.csv,27,0,0,4,normanniles2/normanniles2_2025-10-21-11-34-45.mp4,2025-10-21 11:34:47,2025-10-21 11:35:47,2025-10-21,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-21-17-28-45.mp4,normanniles2_2025-10-21-17-28-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",25,14,11,18.8736,10.8053,8.0683,8.32721202003339,14,25.041736227045075,3105.399859663192,2911.118610801482,0.678363618783357,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-21-17-28-45.mp4_crossings.csv,14,0,1,10,normanniles2/normanniles2_2025-10-21-17-28-45.mp4,2025-10-21 17:28:48,2025-10-21 17:29:48,2025-10-21,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-11-11-45.mp4,normanniles2_2025-10-22-11-11-45.mp4,normanniles2,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",29,19,10,22.3034,14.6251,7.6783,8.300751879699249,12,29.072681704260653,2684.851147342996,2683.1086571189194,0.6773053582734466,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-22-11-11-45.mp4_crossings.csv,26,0,0,3,normanniles2/normanniles2_2025-10-22-11-11-45.mp4,2025-10-22 11:11:46,2025-10-22 11:12:46,2025-10-22,train,light delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-16-24-45.mp4,normanniles2_2025-10-22-16-24-45.mp4,normanniles2,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",23,11,12,17.0822,8.0298,9.0524,10.563909774436093,13,23.05764411027569,1937.4231712139183,2364.842577864148,0.6176334113077054,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-22-16-24-45.mp4_crossings.csv,19,0,0,4,normanniles2/normanniles2_2025-10-22-16-24-45.mp4,2025-10-22 16:24:47,2025-10-22 16:25:47,2025-10-22,train,free flowing,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-16-58-45.mp4,normanniles2_2025-10-22-16-58-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",24,15,9,18.273,11.7768,6.4962,12.064273789649416,18,24.040066777963272,2401.6038884660625,2367.1624032671693,0.682311338422468,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-22-16-58-45.mp4_crossings.csv,19,0,0,5,normanniles2/normanniles2_2025-10-22-16-58-45.mp4,2025-10-22 16:58:47,2025-10-22 16:59:47,2025-10-22,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-23-15-58-45.mp4,normanniles2_2025-10-23-15-58-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",26,16,10,20.0332,12.4041,7.6291,10.535893155258764,15,26.04340567612688,2299.568293455871,2218.8624733266456,0.6753753671666684,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-23-15-58-45.mp4_crossings.csv,24,0,0,2,normanniles2/normanniles2_2025-10-23-15-58-45.mp4,2025-10-23 15:58:45,2025-10-23 15:59:45,2025-10-23,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-11-07-45.mp4,normanniles2_2025-10-24-11-07-45.mp4,normanniles2,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",24,20,4,17.4073,14.2953,3.112,6.433333333333334,13,24.0,2466.23341968912,1976.9503396029404,0.6758124601288935,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-24-11-07-45.mp4_crossings.csv,21,0,0,3,normanniles2/normanniles2_2025-10-24-11-07-45.mp4,2025-10-24 11:07:46,2025-10-24 11:08:46,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-13-10-45.mp4,normanniles2_2025-10-24-13-10-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",24,14,10,18.4248,10.6336,7.7912,9.424040066777962,13,24.040066777963272,3152.173427812223,3928.9223570949393,0.6624452210543719,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-24-13-10-45.mp4_crossings.csv,17,0,1,6,normanniles2/normanniles2_2025-10-24-13-10-45.mp4,2025-10-24 13:10:47,2025-10-24 13:11:47,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-15-05-45.mp4,normanniles2_2025-10-24-15-05-45.mp4,normanniles2,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",22,15,7,16.9461,11.9288,5.0173,6.143692564745196,10,22.05513784461153,2944.541745988577,3044.1378359432697,0.6391887638149523,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-24-15-05-45.mp4_crossings.csv,14,0,0,8,normanniles2/normanniles2_2025-10-24-15-05-45.mp4,2025-10-24 15:05:44,2025-10-24 15:06:44,2025-10-24,train,free flowing,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-17-35-45.mp4,normanniles2_2025-10-24-17-35-45.mp4,normanniles2,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",23,16,7,17.8745,12.3753,5.4992,9.59432387312187,15,23.03839732888147,2184.902905863929,1868.6615797605325,0.6710680685269743,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles2_normanniles2_2025-10-24-17-35-45.mp4_crossings.csv,21,0,0,2,normanniles2/normanniles2_2025-10-24-17-35-45.mp4,2025-10-24 17:35:44,2025-10-24 17:36:44,2025-10-24,train,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-12-47-45.mp4,normanniles3_2025-10-20-12-47-45.mp4,normanniles3,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",36,9,27,26.1058,6.9636,19.1422,13.095238095238097,19,36.090225563909776,1106.0910366826156,1579.759817391157,0.6323128527058274,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-20-12-47-45.mp4_crossings.csv,30,0,0,6,normanniles3/normanniles3_2025-10-20-12-47-45.mp4,2025-10-20 12:47:43,2025-10-20 12:48:43,2025-10-20,train,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-15-04-45.mp4,normanniles3_2025-10-20-15-04-45.mp4,normanniles3,yolo11l.pt,1204,20.0,60.2,960x540,"(48, 270)->(912, 270)",23,9,14,16.5197,7.1579,9.3618,10.100498338870432,15,22.92358803986711,1279.444042430721,2070.5095352105127,0.5882236473415475,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-20-15-04-45.mp4_crossings.csv,15,0,0,8,normanniles3/normanniles3_2025-10-20-15-04-45.mp4,2025-10-20 15:04:42,2025-10-20 15:05:42,2025-10-20,train,light delay,light delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-15-47-45.mp4,normanniles3_2025-10-20-15-47-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",34,8,26,25.7867,6.4739,19.3128,9.708681135225376,13,34.0567612687813,1707.3425328862522,2513.7598971480465,0.6606953594224741,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-20-15-47-45.mp4_crossings.csv,23,0,0,11,normanniles3/normanniles3_2025-10-20-15-47-45.mp4,2025-10-20 15:47:42,2025-10-20 15:48:42,2025-10-20,train,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-17-08-45.mp4,normanniles3_2025-10-20-17-08-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",32,10,22,23.6255,8.4509,15.1746,9.50500834724541,17,32.053422370617696,1723.018705541407,1900.0587093261288,0.6366125423004166,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-20-17-08-45.mp4_crossings.csv,29,0,0,3,normanniles3/normanniles3_2025-10-20-17-08-45.mp4,2025-10-20 17:08:42,2025-10-20 17:09:42,2025-10-20,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-21-07-26-45.mp4,normanniles3_2025-10-21-07-26-45.mp4,normanniles3,yolo11l.pt,1194,20.0,59.7,960x540,"(48, 270)->(912, 270)",1,0,1,0.6242,0.0,0.6242,0.6440536013400335,2,1.0050251256281406,910.0793237971392,1283.9987878015097,0.5461747380378033,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-21-07-26-45.mp4_crossings.csv,1,0,0,0,normanniles3/normanniles3_2025-10-21-07-26-45.mp4,2025-10-21 07:26:44,2025-10-21 07:27:44,2025-10-21,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-21-16-47-45.mp4,normanniles3_2025-10-21-16-47-45.mp4,normanniles3,yolo11l.pt,1199,20.0,59.95,960x540,"(48, 270)->(912, 270)",36,11,25,26.974,9.2026,17.7714,14.619683069224354,22,36.03002502085071,1476.5252438815676,2044.832879534281,0.6380137780243352,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-21-16-47-45.mp4_crossings.csv,27,0,0,9,normanniles3/normanniles3_2025-10-21-16-47-45.mp4,2025-10-21 16:47:43,2025-10-21 16:48:43,2025-10-21,train,heavy delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-22-11-26-45.mp4,normanniles3_2025-10-22-11-26-45.mp4,normanniles3,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",45,5,40,31.3521,3.8986,27.4535,12.76858813700919,18,45.11278195488722,1301.2403166710285,1608.2122642089116,0.6483502660989391,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-22-11-26-45.mp4_crossings.csv,37,0,0,8,normanniles3/normanniles3_2025-10-22-11-26-45.mp4,2025-10-22 11:26:44,2025-10-22 11:27:44,2025-10-22,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-22-17-52-45.mp4,normanniles3_2025-10-22-17-52-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",35,14,21,26.9102,11.5724,15.3377,15.28881469115192,19,35.058430717863104,1603.2623935357065,2304.9815439234967,0.6421672044391861,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-22-17-52-45.mp4_crossings.csv,21,0,0,14,normanniles3/normanniles3_2025-10-22-17-52-45.mp4,2025-10-22 17:52:44,2025-10-22 17:53:44,2025-10-22,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-23-09-19-45.mp4,normanniles3_2025-10-23-09-19-45.mp4,normanniles3,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",6,2,4,3.7962,1.1299,2.6664,2.050125313283208,6,6.015037593984962,773.1071719641402,1085.3563299694702,0.5550925679984574,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-23-09-19-45.mp4_crossings.csv,6,0,0,0,normanniles3/normanniles3_2025-10-23-09-19-45.mp4,2025-10-23 09:19:42,2025-10-23 09:20:42,2025-10-23,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-23-10-32-45.mp4,normanniles3_2025-10-23-10-32-45.mp4,normanniles3,yolo11l.pt,1195,20.0,59.75,960x540,"(48, 270)->(912, 270)",29,7,22,21.9362,5.4335,16.5026,10.799163179916318,16,29.12133891213389,1240.8972491282448,2024.9490681337231,0.6864481792916212,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-23-10-32-45.mp4_crossings.csv,26,0,0,3,normanniles3/normanniles3_2025-10-23-10-32-45.mp4,2025-10-23 10:32:42,2025-10-23 10:33:41,2025-10-23,train,light delay,free flowing,59.0\r\n'
        'normanniles3/normanniles3_2025-10-23-14-45-45.mp4,normanniles3_2025-10-23-14-45-45.mp4,normanniles3,yolo11l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",41,13,28,29.639,9.6917,19.9473,14.40484140233723,19,41.068447412353926,1545.8474242336442,2377.014374450188,0.6265371864249368,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-23-14-45-45.mp4_crossings.csv,29,0,0,12,normanniles3/normanniles3_2025-10-23-14-45-45.mp4,2025-10-23 14:45:41,2025-10-23 14:46:41,2025-10-23,train,light delay,light delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-24-14-10-45.mp4,normanniles3_2025-10-24-14-10-45.mp4,normanniles3,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",25,9,16,18.62,6.9742,11.6459,10.985797827903092,17,25.062656641604008,1425.7158174904944,1945.2484243201816,0.6091450372955871,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles3_normanniles3_2025-10-24-14-10-45.mp4_crossings.csv,19,0,1,5,normanniles3/normanniles3_2025-10-24-14-10-45.mp4,2025-10-24 14:10:43,2025-10-24 14:11:43,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-20-13-05-45.mp4,normanniles4_2025-10-20-13-05-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",31,16,15,22.2741,10.8868,11.3873,10.333333333333334,15,31.0,1792.1123387096775,2263.738544733528,0.6651519145026442,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-20-13-05-45.mp4_crossings.csv,24,0,2,5,normanniles4/normanniles4_2025-10-20-13-05-45.mp4,2025-10-20 13:05:44,2025-10-20 13:06:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-20-16-12-45.mp4,normanniles4_2025-10-20-16-12-45.mp4,normanniles4,yolo11l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",19,14,5,14.6318,10.8871,3.7447,11.532056619483765,15,18.984179850124896,1998.115812274368,2180.8450048500868,0.6765898869245814,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-20-16-12-45.mp4_crossings.csv,14,0,0,5,normanniles4/normanniles4_2025-10-20-16-12-45.mp4,2025-10-20 16:12:44,2025-10-20 16:13:44,2025-10-20,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-22-10-16-45.mp4,normanniles4_2025-10-22-10-16-45.mp4,normanniles4,yolo11l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",12,5,7,8.3862,3.3716,5.0146,2.6644462947543714,5,11.99000832639467,1912.2740625,2257.900694965845,0.707840866263723,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-22-10-16-45.mp4_crossings.csv,10,1,1,0,normanniles4/normanniles4_2025-10-22-10-16-45.mp4,2025-10-22 10:16:44,2025-10-22 10:17:44,2025-10-22,train,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-22-14-45-45.mp4,normanniles4_2025-10-22-14-45-45.mp4,normanniles4,yolo11l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",28,14,14,19.544,9.6797,9.8643,11.138218151540382,15,27.9766860949209,2080.9880391717124,2120.8936755439368,0.6954099037462756,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-22-14-45-45.mp4_crossings.csv,24,0,0,4,normanniles4/normanniles4_2025-10-22-14-45-45.mp4,2025-10-22 14:45:44,2025-10-22 14:46:44,2025-10-22,train,light delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-23-17-00-45.mp4,normanniles4_2025-10-23-17-00-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",28,14,14,20.1058,9.3733,10.7325,13.935833333333331,18,28.0,1695.9792501345453,2844.545175696942,0.6631530468615948,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-23-17-00-45.mp4_crossings.csv,21,0,0,7,normanniles4/normanniles4_2025-10-23-17-00-45.mp4,2025-10-23 17:00:38,2025-10-23 17:01:38,2025-10-23,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-09-58-45.mp4,normanniles4_2025-10-24-09-58-45.mp4,normanniles4,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",5,2,3,3.6537,1.3988,2.2549,1.0100250626566416,5,5.012531328320802,1905.392886683209,1768.4493930832357,0.7413154384467945,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-09-58-45.mp4_crossings.csv,4,0,0,1,normanniles4/normanniles4_2025-10-24-09-58-45.mp4,2025-10-24 09:58:44,2025-10-24 09:59:44,2025-10-24,train,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-10-47-45.mp4,normanniles4_2025-10-24-10-47-45.mp4,normanniles4,yolo11l.pt,1209,20.0,60.45,960x540,"(48, 270)->(912, 270)",26,11,15,18.3957,7.0372,11.3585,8.209263854425144,13,25.806451612903224,1940.391637279597,1945.5724795337185,0.7094069394897153,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-10-47-45.mp4_crossings.csv,25,0,1,0,normanniles4/normanniles4_2025-10-24-10-47-45.mp4,2025-10-24 10:47:44,2025-10-24 10:48:44,2025-10-24,train,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-10-54-45.mp4,normanniles4_2025-10-24-10-54-45.mp4,normanniles4,yolo11l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",27,10,17,18.9538,6.9805,11.9733,12.317460317460318,16,27.06766917293233,1880.569994574064,1918.40540229606,0.6839327245591409,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-10-54-45.mp4_crossings.csv,21,0,0,6,normanniles4/normanniles4_2025-10-24-10-54-45.mp4,2025-10-24 10:54:44,2025-10-24 10:55:44,2025-10-24,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-12-33-45.mp4,normanniles4_2025-10-24-12-33-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",26,6,20,18.6651,4.1886,14.4765,8.620833333333334,13,26.0,1590.4020299661672,1852.1821470221143,0.6994859746511919,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-12-33-45.mp4_crossings.csv,21,1,0,4,normanniles4/normanniles4_2025-10-24-12-33-45.mp4,2025-10-24 12:33:44,2025-10-24 12:34:44,2025-10-24,train,free flowing,light delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-13-10-45.mp4,normanniles4_2025-10-24-13-10-45.mp4,normanniles4,yolo11l.pt,1202,20.0,60.1,960x540,"(48, 270)->(912, 270)",25,12,13,17.4418,8.1338,9.308,12.831114808652249,20,24.958402662229616,1880.4590546586269,2661.502355493274,0.6887122785451173,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-13-10-45.mp4_crossings.csv,17,1,1,6,normanniles4/normanniles4_2025-10-24-13-10-45.mp4,2025-10-24 13:10:44,2025-10-24 13:11:44,2025-10-24,train,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-13-33-45.mp4,normanniles4_2025-10-24-13-33-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",35,20,15,24.1153,12.9061,11.2093,14.864166666666666,22,35.0,2004.3759600829733,2683.5797506747026,0.6572777669397081,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-13-33-45.mp4_crossings.csv,22,0,2,11,normanniles4/normanniles4_2025-10-24-13-33-45.mp4,2025-10-24 13:33:44,2025-10-24 13:34:44,2025-10-24,train,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-14-18-45.mp4,normanniles4_2025-10-24-14-18-45.mp4,normanniles4,yolo11l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",21,10,11,14.3191,6.2748,8.0443,9.428333333333333,13,21.0,2653.9264627894645,3912.470299460328,0.6987381370845818,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_v2/normanniles4_normanniles4_2025-10-24-14-18-45.mp4_crossings.csv,14,0,0,7,normanniles4/normanniles4_2025-10-24-14-18-45.mp4,2025-10-24 14:18:44,2025-10-24 14:19:44,2025-10-24,train,moderate delay,moderate delay,60.0\r\n'
    ),
    'yolov8': (
        'video_path,filename,camera_id,model_name,total_frames,fps,duration_sec,resolution,counting_line,total_unique_vehicles,entering_count,exiting_count,soft_count,soft_entering_count,soft_exiting_count,avg_vehicles_per_frame,max_vehicles_per_frame,flow_rate_per_min,avg_bbox_area,std_bbox_area,avg_confidence,crossing_log,car_count,motorcycle_count,bus_count,truck_count,videos,source_time,target_time,source_date,split,next_enter_rating,congestion_enter_rating,horizon_sec\r\n'
        'normanniles1/normanniles1_2025-10-25-13-21-45.mp4,normanniles1_2025-10-25-13-21-45.mp4,normanniles1,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",40,25,15,29.943,18.9423,11.0006,9.725146198830409,19,40.100250626566414,2437.8927927154023,4899.905120916856,0.684795662945703,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-25-13-21-45.mp4_crossings.csv,33,0,0,7,normanniles1/normanniles1_2025-10-25-13-21-45.mp4,2025-10-25 13:21:44,2025-10-25 13:22:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-14-09-45.mp4,normanniles1_2025-10-25-14-09-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",54,25,29,40.0679,18.2078,21.86,12.410684474123538,17,54.090150250417366,1743.2255178907722,2083.1288678170254,0.7108549551711585,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-25-14-09-45.mp4_crossings.csv,47,0,1,6,normanniles1/normanniles1_2025-10-25-14-09-45.mp4,2025-10-25 14:09:44,2025-10-25 14:10:44,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-03-45.mp4,normanniles1_2025-10-25-17-03-45.mp4,normanniles1,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",22,19,3,15.914,13.7173,2.1967,5.773600668337511,12,22.05513784461153,1320.3536391260309,2733.013894799816,0.6374023558466553,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-25-17-03-45.mp4_crossings.csv,19,0,0,3,normanniles1/normanniles1_2025-10-25-17-03-45.mp4,2025-10-25 17:03:44,2025-10-25 17:04:44,2025-10-25,test,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-46-45.mp4,normanniles1_2025-10-25-17-46-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",37,19,18,26.7357,13.0151,13.7206,11.731218697829716,16,37.06176961602671,1899.4751672121813,2266.9228447217924,0.694193991647814,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-25-17-46-45.mp4_crossings.csv,30,0,1,6,normanniles1/normanniles1_2025-10-25-17-46-45.mp4,2025-10-25 17:46:44,2025-10-25 17:47:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-25-17-55-45.mp4,normanniles1_2025-10-25-17-55-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",43,19,24,32.6578,13.9992,18.6586,13.437395659432386,20,43.07178631051753,1458.9578829668285,2208.2445915278986,0.6984574607414816,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-25-17-55-45.mp4_crossings.csv,39,1,0,3,normanniles1/normanniles1_2025-10-25-17-55-45.mp4,2025-10-25 17:55:44,2025-10-25 17:56:44,2025-10-25,test,heavy delay,light delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-26-13-14-45.mp4,normanniles1_2025-10-26-13-14-45.mp4,normanniles1,yolov8l.pt,1163,20.0,58.15,960x540,"(48, 270)->(912, 270)",32,14,18,22.6261,9.9456,12.6805,7.036973344797937,13,33.01805674978504,1437.3575268817203,2343.9416546045186,0.655526536561454,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-26-13-14-45.mp4_crossings.csv,23,0,0,9,normanniles1/normanniles1_2025-10-26-13-14-45.mp4,2025-10-26 13:14:33,2025-10-26 13:15:42,2025-10-26,test,moderate delay,free flowing,69.0\r\n'
        'normanniles1/normanniles1_2025-10-26-17-16-45.mp4,normanniles1_2025-10-26-17-16-45.mp4,normanniles1,yolov8l.pt,1161,20.0,58.05,960x540,"(48, 270)->(912, 270)",29,19,10,21.2855,14.2815,7.004,6.771748492678725,12,29.97416020671835,933.9629865174256,1611.8534982594,0.6429731404118352,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-26-17-16-45.mp4_crossings.csv,27,0,0,2,normanniles1/normanniles1_2025-10-26-17-16-45.mp4,2025-10-26 17:16:39,2025-10-26 17:17:42,2025-10-26,test,moderate delay,free flowing,63.0\r\n'
        'normanniles1/normanniles1_2025-10-26-17-39-45.mp4,normanniles1_2025-10-26-17-39-45.mp4,normanniles1,yolov8l.pt,1160,20.0,58.0,960x540,"(48, 270)->(912, 270)",38,20,18,27.2881,14.2853,13.0028,9.51896551724138,16,39.310344827586206,1256.199782648071,1905.182731041728,0.6761030305660769,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-26-17-39-45.mp4_crossings.csv,32,0,1,5,normanniles1/normanniles1_2025-10-26-17-39-45.mp4,2025-10-26 17:39:33,2025-10-26 17:40:42,2025-10-26,test,free flowing,light delay,69.0\r\n'
        'normanniles2/normanniles2_2025-10-25-12-44-45.mp4,normanniles2_2025-10-25-12-44-45.mp4,normanniles2,yolov8l.pt,1074,20.0,53.7,960x540,"(48, 270)->(912, 270)",19,13,6,15.6023,10.723,4.8793,5.3472998137802605,10,21.22905027932961,3383.4983458122933,2947.7927802712857,0.6994274501694778,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-25-12-44-45.mp4_crossings.csv,15,0,1,3,normanniles2/normanniles2_2025-10-25-12-44-45.mp4,2025-10-25 12:44:47,2025-10-25 12:45:45,2025-10-25,test,light delay,light delay,58.0\r\n'
        'normanniles2/normanniles2_2025-10-25-13-08-45.mp4,normanniles2_2025-10-25-13-08-45.mp4,normanniles2,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",23,9,14,18.4538,7.4646,10.9892,7.7475,13,23.0,3084.7532537377647,3252.5005570666463,0.7339930418344663,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-25-13-08-45.mp4_crossings.csv,17,0,2,4,normanniles2/normanniles2_2025-10-25-13-08-45.mp4,2025-10-25 13:08:47,2025-10-25 13:09:47,2025-10-25,test,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-14-23-45.mp4,normanniles2_2025-10-25-14-23-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",23,15,8,18.7439,12.1416,6.6022,11.369782971619363,14,23.03839732888147,2325.5644225827764,2240.3242500850984,0.7164975225339617,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-25-14-23-45.mp4_crossings.csv,19,0,0,4,normanniles2/normanniles2_2025-10-25-14-23-45.mp4,2025-10-25 14:23:47,2025-10-25 14:24:47,2025-10-25,test,moderate delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-16-49-45.mp4,normanniles2_2025-10-25-16-49-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",21,10,11,16.1751,8.3075,7.8676,10.41736227045075,17,21.035058430717864,2140.8841346153845,2468.59664354027,0.6721998003060714,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-25-16-49-45.mp4_crossings.csv,14,0,0,7,normanniles2/normanniles2_2025-10-25-16-49-45.mp4,2025-10-25 16:49:48,2025-10-25 16:50:48,2025-10-25,test,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-25-17-44-45.mp4,normanniles2_2025-10-25-17-44-45.mp4,normanniles2,yolov8l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",22,10,12,17.6045,8.3143,9.2902,7.891304347826087,12,22.073578595317727,2306.628417037508,2350.6902138399587,0.7639412295892692,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-25-17-44-45.mp4_crossings.csv,16,1,0,5,normanniles2/normanniles2_2025-10-25-17-44-45.mp4,2025-10-25 17:44:48,2025-10-25 17:45:48,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-06-29-45.mp4,normanniles2_2025-10-26-06-29-45.mp4,normanniles2,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",8,3,5,6.2946,2.6123,3.6823,1.0918964076858817,4,8.020050125313283,3650.7536342769695,2245.968573115681,0.8030185199356207,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-26-06-29-45.mp4_crossings.csv,7,0,0,1,normanniles2/normanniles2_2025-10-26-06-29-45.mp4,2025-10-26 06:29:46,2025-10-26 06:30:46,2025-10-26,test,free flowing,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-14-05-45.mp4,normanniles2_2025-10-26-14-05-45.mp4,normanniles2,yolov8l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",14,5,9,11.0768,4.2058,6.8709,4.614548494983278,9,14.046822742474918,2671.793622032977,2099.5640173288407,0.7228963894445215,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-26-14-05-45.mp4_crossings.csv,9,0,0,5,normanniles2/normanniles2_2025-10-26-14-05-45.mp4,2025-10-26 14:05:47,2025-10-26 14:06:47,2025-10-26,test,light delay,free flowing,60.0\r\n'
        'normanniles2/normanniles2_2025-10-26-15-02-45.mp4,normanniles2_2025-10-26-15-02-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",20,11,9,15.8527,9.0073,6.8454,5.863105175292153,9,20.03338898163606,2567.0099658314352,2415.9808420186014,0.7023312224664965,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-26-15-02-45.mp4_crossings.csv,17,0,1,2,normanniles2/normanniles2_2025-10-26-15-02-45.mp4,2025-10-26 15:02:47,2025-10-26 15:03:47,2025-10-26,test,free flowing,moderate delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-09-33-45.mp4,normanniles3_2025-10-25-09-33-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",10,4,6,7.7409,3.4645,4.2764,0.7587646076794657,3,10.01669449081803,3950.084708470847,2053.4045400616283,0.7784809300100187,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-25-09-33-45.mp4_crossings.csv,9,0,1,0,normanniles3/normanniles3_2025-10-25-09-33-45.mp4,2025-10-25 09:33:44,2025-10-25 09:34:44,2025-10-25,test,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-11-07-45.mp4,normanniles3_2025-10-25-11-07-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",34,10,24,25.1237,8.3544,16.7693,5.297996661101837,9,34.0567612687813,2047.3540255238693,2201.081805849071,0.6998100901277577,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-25-11-07-45.mp4_crossings.csv,24,0,0,10,normanniles3/normanniles3_2025-10-25-11-07-45.mp4,2025-10-25 11:07:44,2025-10-25 11:08:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-13-00-45.mp4,normanniles3_2025-10-25-13-00-45.mp4,normanniles3,yolov8l.pt,770,20.0,38.5,960x540,"(48, 270)->(912, 270)",23,4,19,17.652,3.5431,14.109,11.623376623376624,16,35.84415584415584,1370.7834636871507,1589.9714906390575,0.6447743529485092,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-25-13-00-45.mp4_crossings.csv,21,0,0,2,normanniles3/normanniles3_2025-10-25-13-00-45.mp4,2025-10-25 13:00:39,2025-10-25 13:01:39,2025-10-25,test,heavy delay,heavy delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-25-16-14-45.mp4,normanniles3_2025-10-25-16-14-45.mp4,normanniles3,yolov8l.pt,1202,20.0,60.1,960x540,"(48, 270)->(912, 270)",35,11,24,26.0195,8.8538,17.1657,13.335274542429284,19,34.94176372712146,1154.8180797304883,1788.93095622955,0.6252922660460437,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-25-16-14-45.mp4_crossings.csv,27,1,1,6,normanniles3/normanniles3_2025-10-25-16-14-45.mp4,2025-10-25 16:14:43,2025-10-25 16:15:44,2025-10-25,test,moderate delay,moderate delay,61.0\r\n'
        'normanniles3/normanniles3_2025-10-25-17-20-45.mp4,normanniles3_2025-10-25-17-20-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",43,14,29,33.5904,11.6008,21.9897,13.198664440734555,19,43.07178631051753,1688.853149506704,2239.69676479147,0.6828346574780796,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-25-17-20-45.mp4_crossings.csv,36,0,0,7,normanniles3/normanniles3_2025-10-25-17-20-45.mp4,2025-10-25 17:20:44,2025-10-25 17:21:44,2025-10-25,test,heavy delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-13-15-45.mp4,normanniles3_2025-10-26-13-15-45.mp4,normanniles3,yolov8l.pt,1199,20.0,59.95,960x540,"(48, 270)->(912, 270)",23,7,16,16.8996,5.5139,11.3856,8.249374478732276,15,23.019182652210173,1042.553432413305,1507.6522306757308,0.6499705178682531,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-26-13-15-45.mp4_crossings.csv,20,0,0,3,normanniles3/normanniles3_2025-10-26-13-15-45.mp4,2025-10-26 13:15:43,2025-10-26 13:16:43,2025-10-26,test,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-15-40-45.mp4,normanniles3_2025-10-26-15-40-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",35,9,26,25.9733,6.8583,19.115,10.773789649415694,15,35.058430717863104,1684.2182536608043,1946.5118071966208,0.6530546040021727,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-26-15-40-45.mp4_crossings.csv,27,0,0,8,normanniles3/normanniles3_2025-10-26-15-40-45.mp4,2025-10-26 15:40:43,2025-10-26 15:41:43,2025-10-26,test,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-26-17-30-45.mp4,normanniles3_2025-10-26-17-30-45.mp4,normanniles3,yolov8l.pt,1164,20.0,58.2,960x540,"(48, 270)->(912, 270)",27,13,14,20.9043,10.7221,10.1821,7.859965635738831,17,27.83505154639175,1773.9405399497214,2541.432918928547,0.6611879281601934,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-26-17-30-45.mp4_crossings.csv,24,0,0,3,normanniles3/normanniles3_2025-10-26-17-30-45.mp4,2025-10-26 17:30:39,2025-10-26 17:31:41,2025-10-26,test,free flowing,light delay,62.0\r\n'
        'normanniles4/normanniles4_2025-10-25-12-05-45.mp4,normanniles4_2025-10-25-12-05-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",22,11,11,16.2282,7.8126,8.4157,5.949166666666667,10,22.0,2308.865527384788,2846.586071333185,0.7460649287643044,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-25-12-05-45.mp4_crossings.csv,15,0,0,7,normanniles4/normanniles4_2025-10-25-12-05-45.mp4,2025-10-25 12:05:44,2025-10-25 12:06:44,2025-10-25,test,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-13-51-45.mp4,normanniles4_2025-10-25-13-51-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",25,19,6,18.8229,13.9902,4.8327,12.2925,18,25.0,2071.497186631415,2527.947819530327,0.7161129423513182,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-25-13-51-45.mp4_crossings.csv,19,0,2,4,normanniles4/normanniles4_2025-10-25-13-51-45.mp4,2025-10-25 13:51:44,2025-10-25 13:52:44,2025-10-25,test,heavy delay,moderate delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-14-05-45.mp4,normanniles4_2025-10-25-14-05-45.mp4,normanniles4,yolov8l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",25,13,12,18.1766,8.8132,9.3634,12.020815986677768,17,24.979184013322232,1674.4400498718571,1975.719933777732,0.6701990905149253,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-25-14-05-45.mp4_crossings.csv,18,0,0,7,normanniles4/normanniles4_2025-10-25-14-05-45.mp4,2025-10-25 14:05:44,2025-10-25 14:06:44,2025-10-25,test,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-14-28-45.mp4,normanniles4_2025-10-25-14-28-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",29,12,17,21.456,8.8895,12.5666,11.854166666666666,15,29.0,1798.9005272407733,1961.5963994110764,0.7184712514192233,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-25-14-28-45.mp4_crossings.csv,24,0,0,5,normanniles4/normanniles4_2025-10-25-14-28-45.mp4,2025-10-25 14:28:44,2025-10-25 14:29:44,2025-10-25,test,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-25-17-05-45.mp4,normanniles4_2025-10-25-17-05-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",21,10,11,14.9769,6.4833,8.4936,10.783333333333331,16,21.0,1827.35,2069.069033494793,0.7103158180865313,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-25-17-05-45.mp4_crossings.csv,15,0,2,4,normanniles4/normanniles4_2025-10-25-17-05-45.mp4,2025-10-25 17:05:43,2025-10-25 17:06:43,2025-10-25,test,moderate delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-26-06-23-45.mp4,normanniles4_2025-10-26-06-23-45.mp4,normanniles4,yolov8l.pt,1244,20.0,62.2,960x540,"(48, 270)->(912, 270)",8,5,3,6.0088,3.5923,2.4166,1.1189710610932475,4,7.717041800643086,2028.521551724138,1924.4951635670573,0.7043068126839553,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-26-06-23-45.mp4_crossings.csv,7,0,0,1,normanniles4/normanniles4_2025-10-26-06-23-45.mp4,2025-10-26 06:23:42,2025-10-26 06:24:44,2025-10-26,test,free flowing,free flowing,62.0\r\n'
        'normanniles4/normanniles4_2025-10-26-15-12-45.mp4,normanniles4_2025-10-26-15-12-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",31,18,13,21.3758,11.9536,9.4223,11.025833333333331,16,31.0,1333.5071423172851,1766.3141554649508,0.7029357962023125,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-26-15-12-45.mp4_crossings.csv,25,0,1,5,normanniles4/normanniles4_2025-10-26-15-12-45.mp4,2025-10-26 15:12:44,2025-10-26 15:13:44,2025-10-26,test,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-26-16-15-45.mp4,normanniles4_2025-10-26-16-15-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",23,17,6,16.3075,11.5334,4.7741,10.574166666666668,15,23.0,1679.3936480416107,1745.4124399705397,0.709497784571695,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-26-16-15-45.mp4_crossings.csv,20,0,0,3,normanniles4/normanniles4_2025-10-26-16-15-45.mp4,2025-10-26 16:15:44,2025-10-26 16:16:44,2025-10-26,test,light delay,light delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-08-37-45.mp4,normanniles1_2025-10-20-08-37-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",7,2,5,4.702,1.0302,3.6717,0.8330550918196995,3,7.011686143572621,2400.104208416833,2548.6633290200784,0.6267441519963478,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-20-08-37-45.mp4_crossings.csv,4,0,1,2,normanniles1/normanniles1_2025-10-20-08-37-45.mp4,2025-10-20 08:37:45,2025-10-20 08:38:45,2025-10-20,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-11-18-45.mp4,normanniles1_2025-10-20-11-18-45.mp4,normanniles1,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",41,23,18,30.5028,17.146,13.3568,16.697577276524644,22,41.10275689223057,1667.045429529194,2059.642145127495,0.709626339219968,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-20-11-18-45.mp4_crossings.csv,32,2,1,6,normanniles1/normanniles1_2025-10-20-11-18-45.mp4,2025-10-20 11:18:45,2025-10-20 11:19:45,2025-10-20,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-11-33-45.mp4,normanniles1_2025-10-20-11-33-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",50,23,27,37.8607,16.7017,21.159,15.321666666666667,20,50.0,1504.0871315131078,1802.5233040476023,0.7225931775765805,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-20-11-33-45.mp4_crossings.csv,47,0,1,2,normanniles1/normanniles1_2025-10-20-11-33-45.mp4,2025-10-20 11:33:44,2025-10-20 11:34:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-20-14-02-45.mp4,normanniles1_2025-10-20-14-02-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",32,15,17,23.1779,10.6743,12.5035,8.364166666666666,14,32.0,2008.0988343130416,2674.711249165023,0.6936602885477171,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-20-14-02-45.mp4_crossings.csv,28,0,1,3,normanniles1/normanniles1_2025-10-20-14-02-45.mp4,2025-10-20 14:02:44,2025-10-20 14:03:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-21-08-05-45.mp4,normanniles1_2025-10-21-08-05-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",1,0,1,0.7383,0.0,0.7383,0.0726210350584307,1,1.001669449081803,3921.1724137931033,1712.891574206064,0.7382785349056639,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-21-08-05-45.mp4_crossings.csv,1,0,0,0,normanniles1/normanniles1_2025-10-21-08-05-45.mp4,2025-10-21 08:05:45,2025-10-21 08:06:45,2025-10-21,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-21-10-59-45.mp4,normanniles1_2025-10-21-10-59-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",50,26,24,35.2309,17.3085,17.9224,14.290833333333332,17,50.0,1573.5599160300892,1735.877587391251,0.702409477606507,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-21-10-59-45.mp4_crossings.csv,46,0,0,4,normanniles1/normanniles1_2025-10-21-10-59-45.mp4,2025-10-21 10:59:44,2025-10-21 11:00:45,2025-10-21,train,moderate delay,heavy delay,61.0\r\n'
        'normanniles1/normanniles1_2025-10-21-17-31-45.mp4,normanniles1_2025-10-21-17-31-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",42,17,25,30.3603,11.4725,18.8878,14.266666666666667,19,42.0,1841.766355140187,2160.595578443048,0.6939941543744111,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-21-17-31-45.mp4_crossings.csv,32,0,1,9,normanniles1/normanniles1_2025-10-21-17-31-45.mp4,2025-10-21 17:31:44,2025-10-21 17:32:44,2025-10-21,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-09-03-45.mp4,normanniles1_2025-10-23-09-03-45.mp4,normanniles1,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",4,1,3,2.987,0.7865,2.2005,0.6385642737896494,2,4.006677796327212,2535.964705882353,2732.934661515676,0.7202136114531872,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-23-09-03-45.mp4_crossings.csv,3,0,0,1,normanniles1/normanniles1_2025-10-23-09-03-45.mp4,2025-10-23 09:03:44,2025-10-23 09:04:44,2025-10-23,train,free flowing,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-10-24-45.mp4,normanniles1_2025-10-23-10-24-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",30,9,21,22.8462,6.5284,16.3178,10.560833333333331,17,30.0,1675.7950761461375,2336.7599740716464,0.6944028747977261,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-23-10-24-45.mp4_crossings.csv,21,0,0,9,normanniles1/normanniles1_2025-10-23-10-24-45.mp4,2025-10-23 10:24:44,2025-10-23 10:25:44,2025-10-23,train,moderate delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-12-35-45.mp4,normanniles1_2025-10-23-12-35-45.mp4,normanniles1,yolov8l.pt,1196,20.0,59.8,960x540,"(48, 270)->(912, 270)",43,28,15,30.5383,19.2241,11.3142,10.229096989966555,14,43.1438127090301,1678.6151708353768,2232.829061849477,0.7044880186768335,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-23-12-35-45.mp4_crossings.csv,39,0,0,4,normanniles1/normanniles1_2025-10-23-12-35-45.mp4,2025-10-23 12:35:44,2025-10-23 12:36:44,2025-10-23,train,light delay,free flowing,60.0\r\n'
        'normanniles1/normanniles1_2025-10-23-17-50-45.mp4,normanniles1_2025-10-23-17-50-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",44,19,25,32.415,13.4938,18.9213,12.316666666666666,17,44.0,1747.8242895805142,2716.349799466851,0.6934831956393905,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-23-17-50-45.mp4_crossings.csv,33,1,1,9,normanniles1/normanniles1_2025-10-23-17-50-45.mp4,2025-10-23 17:50:44,2025-10-23 17:51:44,2025-10-23,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles1/normanniles1_2025-10-24-15-54-45.mp4,normanniles1_2025-10-24-15-54-45.mp4,normanniles1,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",48,24,24,34.1269,15.9108,18.2161,14.073333333333334,19,48.0,1939.757579346281,2404.5130128988726,0.6982232261910569,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles1_normanniles1_2025-10-24-15-54-45.mp4_crossings.csv,31,2,2,13,normanniles1/normanniles1_2025-10-24-15-54-45.mp4,2025-10-24 15:54:43,2025-10-24 15:55:44,2025-10-24,train,moderate delay,heavy delay,61.0\r\n'
        'normanniles2/normanniles2_2025-10-20-17-13-45.mp4,normanniles2_2025-10-20-17-13-45.mp4,normanniles2,yolov8l.pt,1191,20.0,59.55,960x540,"(48, 270)->(912, 270)",16,8,8,12.2204,5.9459,6.2745,4.991603694374475,8,16.120906801007557,2361.4489486963835,1842.453292222406,0.6976558989961925,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-20-17-13-45.mp4_crossings.csv,13,0,0,3,normanniles2/normanniles2_2025-10-20-17-13-45.mp4,2025-10-20 17:13:47,2025-10-20 17:14:46,2025-10-20,train,light delay,free flowing,59.0\r\n'
        'normanniles2/normanniles2_2025-10-21-07-06-45.mp4,normanniles2_2025-10-21-07-06-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",1,1,0,0.8055,0.8055,0.0,0.1118530884808013,2,1.001669449081803,5621.970149253732,2150.9154938759216,0.8084790528710208,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-21-07-06-45.mp4_crossings.csv,1,0,0,0,normanniles2/normanniles2_2025-10-21-07-06-45.mp4,2025-10-21 07:06:46,2025-10-21 07:07:46,2025-10-21,train,free flowing,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-21-11-34-45.mp4,normanniles2_2025-10-21-11-34-45.mp4,normanniles2,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",33,18,15,25.5842,13.9749,11.6093,9.347535505430242,13,33.08270676691729,2329.6396460809724,2382.885572743417,0.7341183062007892,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-21-11-34-45.mp4_crossings.csv,28,1,0,4,normanniles2/normanniles2_2025-10-21-11-34-45.mp4,2025-10-21 11:34:47,2025-10-21 11:35:47,2025-10-21,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-21-17-28-45.mp4,normanniles2_2025-10-21-17-28-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",24,15,9,18.6314,11.7835,6.8479,7.025876460767947,13,24.040066777963272,3192.8352144469527,3104.536305037097,0.761814589781809,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-21-17-28-45.mp4_crossings.csv,14,0,0,10,normanniles2/normanniles2_2025-10-21-17-28-45.mp4,2025-10-21 17:28:48,2025-10-21 17:29:48,2025-10-21,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-11-11-45.mp4,normanniles2_2025-10-22-11-11-45.mp4,normanniles2,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",30,19,11,23.4789,14.6433,8.8356,7.555555555555555,12,30.07518796992481,2743.484188412207,2857.040115205936,0.719616043763643,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-22-11-11-45.mp4_crossings.csv,26,0,0,4,normanniles2/normanniles2_2025-10-22-11-11-45.mp4,2025-10-22 11:11:46,2025-10-22 11:12:46,2025-10-22,train,light delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-16-24-45.mp4,normanniles2_2025-10-22-16-24-45.mp4,normanniles2,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",22,11,11,16.9657,8.4589,8.5069,10.054302422723476,14,22.05513784461153,1904.4183631076028,2465.239038164902,0.6514013630178987,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-22-16-24-45.mp4_crossings.csv,20,0,0,2,normanniles2/normanniles2_2025-10-22-16-24-45.mp4,2025-10-22 16:24:47,2025-10-22 16:25:47,2025-10-22,train,free flowing,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-22-16-58-45.mp4,normanniles2_2025-10-22-16-58-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",23,15,8,18.2673,11.9182,6.3491,11.059265442404008,15,23.03839732888147,2565.778322892293,2408.583667795506,0.7173763537060064,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-22-16-58-45.mp4_crossings.csv,16,0,0,7,normanniles2/normanniles2_2025-10-22-16-58-45.mp4,2025-10-22 16:58:47,2025-10-22 16:59:47,2025-10-22,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-23-15-58-45.mp4,normanniles2_2025-10-23-15-58-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",26,16,10,20.452,12.7253,7.7267,10.590150250417365,15,26.04340567612688,2226.2803657286986,2243.646892398116,0.6812416126835151,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-23-15-58-45.mp4_crossings.csv,21,0,0,5,normanniles2/normanniles2_2025-10-23-15-58-45.mp4,2025-10-23 15:58:45,2025-10-23 15:59:45,2025-10-23,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-11-07-45.mp4,normanniles2_2025-10-24-11-07-45.mp4,normanniles2,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",24,20,4,18.1903,14.9105,3.2798,5.920833333333333,13,24.0,2352.590851513019,2103.436994762789,0.7404925448805576,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-24-11-07-45.mp4_crossings.csv,21,0,0,3,normanniles2/normanniles2_2025-10-24-11-07-45.mp4,2025-10-24 11:07:46,2025-10-24 11:08:46,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-13-10-45.mp4,normanniles2_2025-10-24-13-10-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",23,12,11,18.2319,9.2938,8.9381,9.376460767946575,13,23.03839732888147,2975.8240897356004,4039.4144561577,0.6876211367122352,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-24-13-10-45.mp4_crossings.csv,18,0,1,4,normanniles2/normanniles2_2025-10-24-13-10-45.mp4,2025-10-24 13:10:47,2025-10-24 13:11:47,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-15-05-45.mp4,normanniles2_2025-10-24-15-05-45.mp4,normanniles2,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",23,16,7,18.4151,13.1025,5.3126,5.975772765246449,10,23.05764411027569,3093.3696351181325,3081.9155510521773,0.686759374831088,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-24-15-05-45.mp4_crossings.csv,14,0,0,9,normanniles2/normanniles2_2025-10-24-15-05-45.mp4,2025-10-24 15:05:44,2025-10-24 15:06:44,2025-10-24,train,free flowing,moderate delay,60.0\r\n'
        'normanniles2/normanniles2_2025-10-24-17-35-45.mp4,normanniles2_2025-10-24-17-35-45.mp4,normanniles2,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",22,15,7,17.6066,11.9978,5.6088,8.675292153589316,14,22.036727879799667,2060.245068796305,1977.7300937096747,0.7335506982902074,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles2_normanniles2_2025-10-24-17-35-45.mp4_crossings.csv,19,0,0,3,normanniles2/normanniles2_2025-10-24-17-35-45.mp4,2025-10-24 17:35:44,2025-10-24 17:36:44,2025-10-24,train,light delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-12-47-45.mp4,normanniles3_2025-10-20-12-47-45.mp4,normanniles3,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",35,9,26,25.5713,7.0986,18.4727,11.709273182957393,18,35.08771929824562,1168.566424086758,1661.9391148855902,0.651878483421233,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-20-12-47-45.mp4_crossings.csv,28,0,0,7,normanniles3/normanniles3_2025-10-20-12-47-45.mp4,2025-10-20 12:47:43,2025-10-20 12:48:43,2025-10-20,train,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-15-04-45.mp4,normanniles3_2025-10-20-15-04-45.mp4,normanniles3,yolov8l.pt,1204,20.0,60.2,960x540,"(48, 270)->(912, 270)",22,9,13,15.6002,7.1432,8.457,9.854651162790695,16,21.92691029900332,1295.3483354403709,2129.245060868129,0.6039034276104933,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-20-15-04-45.mp4_crossings.csv,15,0,1,6,normanniles3/normanniles3_2025-10-20-15-04-45.mp4,2025-10-20 15:04:42,2025-10-20 15:05:42,2025-10-20,train,light delay,light delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-15-47-45.mp4,normanniles3_2025-10-20-15-47-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",36,8,28,27.7231,6.609,21.1141,10.165275459098496,14,36.060100166944906,1647.7569387419935,2500.115240282656,0.672721680380388,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-20-15-47-45.mp4_crossings.csv,25,0,0,11,normanniles3/normanniles3_2025-10-20-15-47-45.mp4,2025-10-20 15:47:42,2025-10-20 15:48:42,2025-10-20,train,moderate delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-20-17-08-45.mp4,normanniles3_2025-10-20-17-08-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",33,10,23,24.8344,8.4917,16.3427,8.7220367278798,17,33.0550918196995,1839.2759115704853,1970.7541855995355,0.6821963876872293,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-20-17-08-45.mp4_crossings.csv,26,0,0,7,normanniles3/normanniles3_2025-10-20-17-08-45.mp4,2025-10-20 17:08:42,2025-10-20 17:09:42,2025-10-20,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-21-07-26-45.mp4,normanniles3_2025-10-21-07-26-45.mp4,normanniles3,yolov8l.pt,1194,20.0,59.7,960x540,"(48, 270)->(912, 270)",2,0,2,1.264,0.0,1.264,0.568676716917923,2,2.010050251256281,1141.142857142857,1580.1296134187949,0.645750150907461,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-21-07-26-45.mp4_crossings.csv,2,0,0,0,normanniles3/normanniles3_2025-10-21-07-26-45.mp4,2025-10-21 07:26:44,2025-10-21 07:27:44,2025-10-21,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-21-16-47-45.mp4,normanniles3_2025-10-21-16-47-45.mp4,normanniles3,yolov8l.pt,1199,20.0,59.95,960x540,"(48, 270)->(912, 270)",36,11,25,27.6997,9.4672,18.2325,14.485404503753127,22,36.03002502085071,1491.6452095808384,2109.268252066752,0.6410260244385472,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-21-16-47-45.mp4_crossings.csv,26,0,0,10,normanniles3/normanniles3_2025-10-21-16-47-45.mp4,2025-10-21 16:47:43,2025-10-21 16:48:43,2025-10-21,train,heavy delay,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-22-11-26-45.mp4,normanniles3_2025-10-22-11-26-45.mp4,normanniles3,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",44,5,39,31.818,3.9575,27.8605,12.908939014202172,19,44.11027568922306,1269.9779963758738,1581.5260220866826,0.6688814463266014,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-22-11-26-45.mp4_crossings.csv,37,0,0,7,normanniles3/normanniles3_2025-10-22-11-26-45.mp4,2025-10-22 11:26:44,2025-10-22 11:27:44,2025-10-22,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-22-17-52-45.mp4,normanniles3_2025-10-22-17-52-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",37,14,23,29.1372,11.6788,17.4583,14.715358931552588,19,37.06176961602671,1657.140166770662,2305.912498851513,0.6617998739768836,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-22-17-52-45.mp4_crossings.csv,24,0,0,13,normanniles3/normanniles3_2025-10-22-17-52-45.mp4,2025-10-22 17:52:44,2025-10-22 17:53:44,2025-10-22,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-23-09-19-45.mp4,normanniles3_2025-10-23-09-19-45.mp4,normanniles3,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",6,2,4,4.2578,1.2489,3.0088,1.9924812030075187,6,6.015037593984962,789.988679245283,1108.8498127381213,0.6122700881861166,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-23-09-19-45.mp4_crossings.csv,6,0,0,0,normanniles3/normanniles3_2025-10-23-09-19-45.mp4,2025-10-23 09:19:42,2025-10-23 09:20:42,2025-10-23,train,free flowing,free flowing,60.0\r\n'
        'normanniles3/normanniles3_2025-10-23-10-32-45.mp4,normanniles3_2025-10-23-10-32-45.mp4,normanniles3,yolov8l.pt,1195,20.0,59.75,960x540,"(48, 270)->(912, 270)",29,6,23,21.7999,4.6461,17.1538,10.6,14,29.12133891213389,1260.1874161206283,2015.1963530743683,0.6771029521615369,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-23-10-32-45.mp4_crossings.csv,27,0,0,2,normanniles3/normanniles3_2025-10-23-10-32-45.mp4,2025-10-23 10:32:42,2025-10-23 10:33:41,2025-10-23,train,light delay,free flowing,59.0\r\n'
        'normanniles3/normanniles3_2025-10-23-14-45-45.mp4,normanniles3_2025-10-23-14-45-45.mp4,normanniles3,yolov8l.pt,1198,20.0,59.9,960x540,"(48, 270)->(912, 270)",38,12,26,28.7526,9.52,19.2326,14.802170283806344,19,38.06343906510852,1747.9833079569164,2522.49113195714,0.6291559633790441,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-23-14-45-45.mp4_crossings.csv,30,0,0,8,normanniles3/normanniles3_2025-10-23-14-45-45.mp4,2025-10-23 14:45:41,2025-10-23 14:46:41,2025-10-23,train,light delay,light delay,60.0\r\n'
        'normanniles3/normanniles3_2025-10-24-14-10-45.mp4,normanniles3_2025-10-24-14-10-45.mp4,normanniles3,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",24,9,15,18.2302,7.4371,10.7931,11.570593149540516,17,24.06015037593985,1420.9098194945848,2001.6080113431933,0.6060413844947996,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles3_normanniles3_2025-10-24-14-10-45.mp4_crossings.csv,18,0,1,5,normanniles3/normanniles3_2025-10-24-14-10-45.mp4,2025-10-24 14:10:43,2025-10-24 14:11:43,2025-10-24,train,moderate delay,light delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-20-13-05-45.mp4,normanniles4_2025-10-20-13-05-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",30,15,15,22.3114,10.5423,11.7691,9.8375,13,30.0,1695.8426937738247,2221.494994002082,0.6910685413207406,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-20-13-05-45.mp4_crossings.csv,22,0,2,6,normanniles4/normanniles4_2025-10-20-13-05-45.mp4,2025-10-20 13:05:44,2025-10-20 13:06:44,2025-10-20,train,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-20-16-12-45.mp4,normanniles4_2025-10-20-16-12-45.mp4,normanniles4,yolov8l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",21,15,6,16.3368,11.5267,4.8101,11.32639467110741,15,20.98251457119068,1979.8359920605749,2146.1063381953345,0.693997321052417,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-20-16-12-45.mp4_crossings.csv,16,0,0,5,normanniles4/normanniles4_2025-10-20-16-12-45.mp4,2025-10-20 16:12:44,2025-10-20 16:13:44,2025-10-20,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-22-10-16-45.mp4,normanniles4_2025-10-22-10-16-45.mp4,normanniles4,yolov8l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",12,5,7,8.651,3.4689,5.1822,3.0141548709408825,5,11.99000832639467,1722.021270718232,2240.979300493015,0.722400034908514,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-22-10-16-45.mp4_crossings.csv,9,1,1,1,normanniles4/normanniles4_2025-10-22-10-16-45.mp4,2025-10-22 10:16:44,2025-10-22 10:17:44,2025-10-22,train,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-22-14-45-45.mp4,normanniles4_2025-10-22-14-45-45.mp4,normanniles4,yolov8l.pt,1201,20.0,60.05,960x540,"(48, 270)->(912, 270)",28,14,14,20.1081,9.8369,10.2712,11.522897585345545,16,27.9766860949209,2029.9030999349663,2131.544447542417,0.7208313310696659,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-22-14-45-45.mp4_crossings.csv,24,0,0,4,normanniles4/normanniles4_2025-10-22-14-45-45.mp4,2025-10-22 14:45:44,2025-10-22 14:46:44,2025-10-22,train,light delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-23-17-00-45.mp4,normanniles4_2025-10-23-17-00-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",28,15,13,20.8701,10.5463,10.3239,13.639166666666666,18,28.0,1737.8985153051872,2900.0297816455445,0.676969572130356,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-23-17-00-45.mp4_crossings.csv,21,0,0,7,normanniles4/normanniles4_2025-10-23-17-00-45.mp4,2025-10-23 17:00:38,2025-10-23 17:01:38,2025-10-23,train,heavy delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-09-58-45.mp4,normanniles4_2025-10-24-09-58-45.mp4,normanniles4,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",5,2,3,3.8263,1.412,2.4143,1.1461988304093569,5,5.012531328320802,1709.5371720116618,1775.91130531952,0.7404268060922188,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-09-58-45.mp4_crossings.csv,4,0,0,1,normanniles4/normanniles4_2025-10-24-09-58-45.mp4,2025-10-24 09:58:44,2025-10-24 09:59:44,2025-10-24,train,free flowing,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-10-47-45.mp4,normanniles4_2025-10-24-10-47-45.mp4,normanniles4,yolov8l.pt,1209,20.0,60.45,960x540,"(48, 270)->(912, 270)",25,10,15,17.8802,6.5827,11.2975,8.37799834574028,13,24.813895781637715,1686.0038503307337,1913.9905029182137,0.7328219308648161,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-10-47-45.mp4_crossings.csv,24,0,0,1,normanniles4/normanniles4_2025-10-24-10-47-45.mp4,2025-10-24 10:47:44,2025-10-24 10:48:44,2025-10-24,train,light delay,free flowing,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-10-54-45.mp4,normanniles4_2025-10-24-10-54-45.mp4,normanniles4,yolov8l.pt,1197,20.0,59.85,960x540,"(48, 270)->(912, 270)",26,10,16,19.0594,7.25,11.8095,11.663324979114453,15,26.06516290726817,1717.5622806389226,1894.0999044319512,0.7260390196352329,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-10-54-45.mp4_crossings.csv,22,0,0,4,normanniles4/normanniles4_2025-10-24-10-54-45.mp4,2025-10-24 10:54:44,2025-10-24 10:55:44,2025-10-24,train,heavy delay,moderate delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-12-33-45.mp4,normanniles4_2025-10-24-12-33-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",26,6,20,19.24,4.187,15.053,8.46,12,26.0,1621.483353033885,1886.707821675685,0.7196030839033707,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-12-33-45.mp4_crossings.csv,19,1,0,6,normanniles4/normanniles4_2025-10-24-12-33-45.mp4,2025-10-24 12:33:44,2025-10-24 12:34:44,2025-10-24,train,free flowing,light delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-13-10-45.mp4,normanniles4_2025-10-24-13-10-45.mp4,normanniles4,yolov8l.pt,1202,20.0,60.1,960x540,"(48, 270)->(912, 270)",25,12,13,17.831,8.2118,9.6192,12.494176372712149,20,24.958402662229616,1899.10334265548,2727.3156555519554,0.6990324756742528,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-13-10-45.mp4_crossings.csv,16,1,1,7,normanniles4/normanniles4_2025-10-24-13-10-45.mp4,2025-10-24 13:10:44,2025-10-24 13:11:44,2025-10-24,train,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-13-33-45.mp4,normanniles4_2025-10-24-13-33-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",36,22,14,24.8637,14.3676,10.4962,13.135,19,36.0,2198.3023727953305,2746.3639462836536,0.6923491288914325,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-13-33-45.mp4_crossings.csv,22,0,2,12,normanniles4/normanniles4_2025-10-24-13-33-45.mp4,2025-10-24 13:33:44,2025-10-24 13:34:44,2025-10-24,train,moderate delay,heavy delay,60.0\r\n'
        'normanniles4/normanniles4_2025-10-24-14-18-45.mp4,normanniles4_2025-10-24-14-18-45.mp4,normanniles4,yolov8l.pt,1200,20.0,60.0,960x540,"(48, 270)->(912, 270)",20,9,11,14.8515,6.5035,8.348,8.865833333333333,12,20.0,2579.514428047749,3629.164408397656,0.7296268872924402,/content/drive/MyDrive/CP-1 Dataset/crossing_logs_yolov8_v2/normanniles4_normanniles4_2025-10-24-14-18-45.mp4_crossings.csv,15,0,0,5,normanniles4/normanniles4_2025-10-24-14-18-45.mp4,2025-10-24 14:18:44,2025-10-24 14:19:44,2025-10-24,train,moderate delay,moderate delay,60.0\r\n'
    ),
}
TRACKER_SETTINGS = {'tracker_type': 'bytetrack', 'track_high_thresh': 0.4, 'track_low_thresh': 0.05, 'new_track_thresh': 0.5, 'track_buffer': 90, 'match_thresh': 0.8, 'gmc_method': 'sparseOptFlow', 'proximity_thresh': 0.5, 'appearance_thresh': 0.25, 'with_reid': False, 'fuse_score': True}
PROCESSOR_CODE = '# --- YOLO11 + ByteTrack: independent clips, reproducible crossings ---\nimport copy\nimport json\nimport math\nimport re\nfrom collections import defaultdict\nfrom datetime import datetime, timezone\n\nGLOBAL_TRACKING_STATE = {}\nJOURNAL_RESULT_CONTEXT = None\n\ndef unsharp_mask(image, kernel_size=(5, 5), sigma=1.0, amount=1.5):\n    blurred = cv2.GaussianBlur(image, kernel_size, sigma)\n    return cv2.addWeighted(image, 1 + amount, blurred, -amount, 0)\n\ndef get_adaptive_params(width, height):\n    pixels = width * height\n    if pixels < 854 * 480:\n        return {"apply_sharpen": True, "imgsz": 1280, "conf": 0.05, "line_thick": 1}\n    if pixels <= 1920 * 1080:\n        return {"apply_sharpen": False, "imgsz": 1280, "conf": 0.05, "line_thick": 2}\n    return {"apply_sharpen": False, "imgsz": 1920, "conf": 0.05, "line_thick": 4}\n\ndef segment_intersection(a, b, c, d):\n    def cross(p, q, r):\n        return (q[0] - p[0]) * (r[1] - p[1]) - (q[1] - p[1]) * (r[0] - p[0])\n    ab_c, ab_d = cross(a, b, c), cross(a, b, d)\n    cd_a, cd_b = cross(c, d, a), cross(c, d, b)\n    return ab_c * ab_d < 0 and cd_a * cd_b < 0\n\ndef line_side(a, b, point):\n    return (b[0] - a[0]) * (point[1] - a[1]) - (b[1] - a[1]) * (point[0] - a[0])\n\ndef crossing_direction(a, b, old, new):\n    line_x, line_y = b[0] - a[0], b[1] - a[1]\n    move_x, move_y = new[0] - old[0], new[1] - old[1]\n    return "exiting" if line_x * move_y - line_y * move_x > 0 else "entering"\n\ndef camera_id_from_path(video_path):\n    return os.path.basename(video_path).split("_", 1)[0].lower()\n\ndef line_for_video(video_path, width, height):\n    normalized = COUNTING_LINES.get(camera_id_from_path(video_path), COUNTING_LINES["default"])\n    return tuple((int(round(x * width)), int(round(y * height))) for x, y in normalized)\n\ndef reset_tracker_for_new_clip(model):\n    """ByteTrack\'s IDs/history must not carry into an unrelated video."""\n    predictor = getattr(model, "predictor", None)\n    trackers = getattr(predictor, "trackers", None)\n    if trackers:\n        for tracker in trackers:\n            tracker.reset()\n\ndef generate_heatmap(points, width, height, background_frame):\n    if not points or background_frame is None:\n        return background_frame\n    heat = np.zeros((height, width), dtype=np.float32)\n    for x, y in points:\n        if 0 <= x < width and 0 <= y < height:\n            heat[y, x] += 1\n    heat = cv2.GaussianBlur(heat, (15, 15), 0)\n    if heat.max() <= 0:\n        return background_frame\n    colored = cv2.applyColorMap(np.uint8(255 * heat / heat.max()), cv2.COLORMAP_JET)\n    result = background_frame.copy()\n    blended = cv2.addWeighted(colored, 0.6, background_frame, 0.4, 0)\n    result[heat > 0] = blended[heat > 0]\n    return result\n\ndef process_video_with_tracking(video_path, model, max_frames=None, show_sample=False):\n    """Process one full clip; return features, sample, heatmap or three Nones if interrupted."""\n    global GLOBAL_TRACKING_STATE\n    video_key = make_video_key(video_path)\n    cap = cv2.VideoCapture(video_path)\n    if not cap.isOpened():\n        cap.release()\n        raise OSError(f"Cannot open video: {video_path}")\n    fps = float(cap.get(cv2.CAP_PROP_FPS))\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    if width <= 0 or height <= 0:\n        cap.release()\n        raise ValueError(f"Invalid video dimensions: {video_path}")\n    params = get_adaptive_params(width, height)\n    pt_a, pt_b = line_for_video(video_path, width, height)\n    reset_tracker_for_new_clip(model)\n\n    counts = {direction: defaultdict(int) for direction in ("entering", "exiting")}\n    counted_ids, track_history, last_moved_frame = set(), {}, {}\n    track_confidences = defaultdict(list)\n    frame_data, all_bbox_areas, all_confidences = [], [], []\n    observations, crossings, points = [], [], []\n    sample_frame = background_frame = None\n    frame_idx = 0\n    max_frames = (total_frames if total_frames > 0 else float("inf")) if max_frames is None else max_frames\n    try:\n        while cap.isOpened() and frame_idx < max_frames:\n            ok, frame = cap.read()\n            if not ok:\n                break\n            if background_frame is None:\n                background_frame = frame.copy()\n            processed = unsharp_mask(frame) if params["apply_sharpen"] else frame\n            # persist=True is valid only for successive frames *within this clip*.\n            result = model.track(\n                processed, persist=True, tracker=TRACKER_CONFIG_PATH,\n                imgsz=params["imgsz"], conf=params["conf"], verbose=False,\n                classes=list(VEHICLE_CLASSES)\n            )[0]\n            annotated = processed.copy() if show_sample else None\n            if annotated is not None:\n                cv2.line(annotated, pt_a, pt_b, (0, 255, 255), params["line_thick"])\n            frame_vehicles = 0\n            if result.boxes is not None and result.boxes.id is not None:\n                for box in result.boxes:\n                    class_id = int(box.cls[0])\n                    track_id = int(box.id[0])\n                    confidence = float(box.conf[0])\n                    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())\n                    center = ((x1 + x2) // 2, (y1 + y2) // 2)\n                    class_name = VEHICLE_CLASSES.get(class_id, "unknown")\n                    frame_vehicles += 1\n                    all_bbox_areas.append(max(0, x2 - x1) * max(0, y2 - y1))\n                    all_confidences.append(confidence)\n                    track_confidences[track_id].append(confidence)\n                    points.append(center)\n                    observations.append({\n                        "frame_idx": frame_idx, "track_id": track_id,\n                        "vehicle_class": class_name, "confidence": confidence,\n                        "cx": center[0], "cy": center[1],\n                        "x1": x1, "y1": y1, "x2": x2, "y2": y2,\n                    })\n                    previous = track_history.get(track_id)\n                    is_stopped = False\n                    if previous is None or math.hypot(center[0] - previous[0], center[1] - previous[1]) > 5:\n                        last_moved_frame[track_id] = frame_idx\n                    elif fps > 0 and frame_idx - last_moved_frame.get(track_id, frame_idx) > 3 * fps:\n                        is_stopped = True\n                    if previous is not None and track_id not in counted_ids:\n                        if segment_intersection(pt_a, pt_b, previous, center):\n                            direction = crossing_direction(pt_a, pt_b, previous, center)\n                            counts[direction][class_name] += 1\n                            counted_ids.add(track_id)\n                            crossings.append({\n                                "frame_idx": frame_idx, "track_id": track_id,\n                                "vehicle_class": class_name, "direction": direction,\n                                "crossing_confidence": confidence,\n                                "previous_cx": previous[0], "previous_cy": previous[1],\n                                "cx": center[0], "cy": center[1],\n                            })\n                    # Preserve the last off-line position when a center lands exactly\n                    # on the tripwire; count once it actually reaches the other side.\n                    if line_side(pt_a, pt_b, center) != 0:\n                        track_history[track_id] = center\n                    if annotated is not None:\n                        color = (0, 0, 255) if is_stopped else (255, 0, 0)\n                        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, params["line_thick"])\n                        cv2.putText(annotated, f"{class_name} #{track_id}", (x1, max(15, y1 - 8)),\n                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)\n                        if is_stopped:\n                            cv2.putText(annotated, "STOPPED VEHICLE", (x1, max(25, y1 - 25)),\n                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)\n            frame_data.append({"frame_idx": frame_idx, "vehicles_in_frame": frame_vehicles})\n            if annotated is not None and (frame_idx == total_frames // 2 or sample_frame is None):\n                sample_frame = annotated.copy()\n            frame_idx += 1\n    except KeyboardInterrupt:\n        print(f"Interrupted at frame {frame_idx}; no partial results saved. Rerun this clip from frame zero.")\n        return None, None, None\n    finally:\n        cap.release()\n\n    if frame_idx == 0:\n        raise ValueError(f"No frames read: {video_path}")\n    if total_frames > 0 and frame_idx != total_frames:\n        raise ValueError(f"Incomplete clip: {frame_idx}/{total_frames} frames")\n    for event in crossings:\n        history = track_confidences[event["track_id"]]\n        event["mean_track_confidence"] = float(np.mean(history))\n        event["track_confidence_history"] = json.dumps(history)\n    soft_entering = sum(e["mean_track_confidence"] for e in crossings if e["direction"] == "entering")\n    soft_exiting = sum(e["mean_track_confidence"] for e in crossings if e["direction"] == "exiting")\n    hard_entering = sum(counts["entering"].values())\n    hard_exiting = sum(counts["exiting"].values())\n    duration = frame_idx / fps if fps > 0 else 0.0\n    densities = [row["vehicles_in_frame"] for row in frame_data]\n    safe_key = re.sub(r"[^A-Za-z0-9_.-]+", "_", video_key)\n    log_prefix = os.path.join(RUN_LOG_DIR, safe_key)\n    features = {\n        "video_path": video_key, "filename": os.path.basename(video_path),\n        "camera_id": camera_id_from_path(video_path), "model_name": YOLO_MODEL_NAME,\n        "total_frames": frame_idx, "fps": fps, "duration_sec": round(duration, 3),\n        "resolution": f"{width}x{height}", "counting_line": f"{pt_a}->{pt_b}",\n        "total_unique_vehicles": hard_entering + hard_exiting,\n        "entering_count": hard_entering, "exiting_count": hard_exiting,\n        "soft_count": round(soft_entering + soft_exiting, 4),\n        "soft_entering_count": round(soft_entering, 4),\n        "soft_exiting_count": round(soft_exiting, 4),\n        "avg_vehicles_per_frame": float(np.mean(densities)),\n        "max_vehicles_per_frame": int(max(densities)),\n        "flow_rate_per_min": (hard_entering + hard_exiting) * 60 / duration if duration else 0,\n        "avg_bbox_area": float(np.mean(all_bbox_areas)) if all_bbox_areas else 0,\n        "std_bbox_area": float(np.std(all_bbox_areas)) if all_bbox_areas else 0,\n        "avg_confidence": float(np.mean(all_confidences)) if all_confidences else 0,\n        "crossing_log": log_prefix + "_crossings.csv",\n    }\n    for class_name in VEHICLE_CLASSES.values():\n        features[f"{class_name}_count"] = counts["entering"][class_name] + counts["exiting"][class_name]\n    manifest = {\n        "video_path": video_key, "processed_at_utc": datetime.now(timezone.utc).isoformat(),\n        "model_weights": YOLO_MODEL_NAME, "tracker_config_path": TRACKER_CONFIG_PATH,\n        "tracker_config": custom_config, "ultralytics_version": __import__("ultralytics").__version__,\n        "counting_line_pixels": [pt_a, pt_b], "adaptive_parameters": params,\n        "frames_processed": frame_idx, "fps": fps, "hard_count": hard_entering + hard_exiting,\n        "soft_count_definition": "sum of mean YOLO confidences per crossed track",\n        "soft_count": soft_entering + soft_exiting,\n        "crossings_file": log_prefix + "_crossings.csv",\n        "observations_file": log_prefix + "_observations.csv",\n    }\n    pd.DataFrame(crossings, columns=[\n        "frame_idx", "track_id", "vehicle_class", "direction", "crossing_confidence",\n        "previous_cx", "previous_cy", "cx", "cy", "mean_track_confidence",\n        "track_confidence_history"\n    ]).to_csv(manifest["crossings_file"], index=False)\n    pd.DataFrame(observations, columns=[\n        "frame_idx", "track_id", "vehicle_class", "confidence", "cx", "cy", "x1", "y1", "x2", "y2"\n    ]).to_csv(manifest["observations_file"], index=False)\n    with open(log_prefix + "_manifest.json", "w", encoding="utf-8") as handle:\n        json.dump(manifest, handle, indent=2)\n    GLOBAL_TRACKING_STATE = {\n        "video_path": video_key, "counts": counts, "frame_data": frame_data,\n        "all_confidences": all_confidences, "all_bbox_areas": all_bbox_areas,\n        "crossings": crossings, "features": features,\n    }\n    return features, sample_frame, generate_heatmap(points, width, height, background_frame)\n\n'
historical = pd.read_csv(io.StringIO(CACHED_FEATURES[DETECTOR]))
expected_model = "yolov8l.pt" if DETECTOR == "yolov8" else "yolo11l.pt"
if not historical.model_name.eq(expected_model).all():
    raise ValueError("Cached feature model does not match DETECTOR.")
print("Detector:", DETECTOR, "| Loaded completed clips:", len(historical))

In [ ]:
# Colab orchestration: labels are matched to video paths, never read from MP4 metadata.
from google.colab import drive
drive.mount('/content/drive')
drive_base = Path(DRIVE_BASE)
if not drive_base.is_dir():
    raise FileNotFoundError('CP-1 Dataset is not accessible. Check the mounted account and My Drive shortcut.')
output_root = drive_base/'Forecasting_v3'/DETECTOR
if RUN_MODE == 'expanded':
    output_root = output_root/EXPERIMENT_NAME
output_root.mkdir(parents=True, exist_ok=True)
features = None

if RUN_MODE == 'expanded':
    import os
    import time
    available = {}
    for camera in ['normanniles1','normanniles2','normanniles3','normanniles4']:
        folder = drive_base/camera
        if not folder.is_dir():
            raise FileNotFoundError('Camera folder is inaccessible: '+str(folder))
        print('Scanning video filenames:', camera, flush=True)
        for directory, _, names in os.walk(folder):
            for name in names:
                if name.lower().endswith('.mp4'):
                    video = Path(directory)/name
                    available[video.relative_to(drive_base).as_posix()] = str(video)
    label_path = drive_base/LABELS_RELATIVE_PATH
    if not label_path.is_file():
        raise FileNotFoundError('Label CSV is missing: '+str(label_path)+'. Copy the project Train.csv here or edit LABELS_RELATIVE_PATH.')
    labels = pd.read_csv(label_path)
    coverage, missing_labels, unlabelled_videos = dataset_label_audit(labels, available)
    timing_issues = label_timing_issues(labels)
    print('Dataset/label match:')
    display(coverage)
    plan = expansion_plan(labels, available, historical,
                          TRAIN_CLIPS_PER_CAMERA_DATE, TEST_CLIPS_PER_CAMERA_DATE)
    sampling_limits = {'train_per_camera_date':TRAIN_CLIPS_PER_CAMERA_DATE,
                       'test_per_camera_date':TEST_CLIPS_PER_CAMERA_DATE}
    plan, previous_contract = resume_saved_plan(plan,labels,available,historical,
                                                output_root,sampling_limits)
    print('Selected clips (training labels do not control sampling):')
    print(plan.groupby(['split','camera_id']).size())
    print('Total selected:',len(plan),'| Train:',int(plan.split.eq('train').sum()),
          '| Evaluation:',int(plan.split.eq('test').sum()))
    coverage.to_csv(output_root/'dataset_inventory.csv', index=False)
    missing_labels.to_csv(output_root/'labelled_videos_missing_from_drive.csv', index=False)
    unlabelled_videos.to_csv(output_root/'videos_without_labels.csv', index=False)
    timing_issues.to_csv(output_root/'invalid_label_timing.csv', index=False)
    print('Invalid label timings excluded from source/target selection:',len(timing_issues))
    preview = output_root/'preflight'
    preview.mkdir(exist_ok=True)
    plan.to_csv(preview/'proposed_pairs.csv', index=False)
    (preview/'SUMMARY.json').write_text(json.dumps({
        'detector':DETECTOR, 'mode':RUN_MODE, 'experiment':EXPERIMENT_NAME,
        'label_rows':len(labels), 'accessible_videos':len(available),
        'matched_videos':int(coverage.matched_videos.sum()),
        'invalid_label_timings':len(timing_issues),
        'train_clips':int(plan.split.eq('train').sum()),
        'evaluation_clips':int(plan.split.eq('test').sum()),
        'new_independent_dates':False}, indent=2))
    RESULT_DIR = output_root/'evaluation'
    EVALUATION_KIND = 'new_clips_on_previously_examined_dates'

    if PLAN_ONLY:
        print('Preflight saved:', preview)
        print('No videos processed. Set PLAN_ONLY=False and run from the top to start extraction.')
    else:
        import subprocess, sys
        saved_version = (previous_contract or {}).get('configuration',{}).get('ultralytics_version')
        requirement = 'ultralytics=='+saved_version if saved_version else 'ultralytics>=8.3.0'
        if saved_version:
            print('Resuming with saved Ultralytics version:',saved_version)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',requirement])
        loaded_ultralytics = sys.modules.get('ultralytics')
        if (saved_version and loaded_ultralytics is not None
                and getattr(loaded_ultralytics,'__version__',saved_version) != saved_version):
            raise RuntimeError('The saved Ultralytics version is installed, but another version is already loaded. Restart the Colab kernel, then Run All with unchanged settings.')
        import cv2, torch, yaml
        from ultralytics import YOLO
        if not torch.cuda.is_available():
            raise RuntimeError('Select a Colab GPU (for example T4), reconnect, and run from the top. The preflight plan is saved.')
        YOLO_MODEL_NAME = 'yolo11l.pt' if DETECTOR == 'yolo11' else 'yolov8l.pt'
        VEHICLE_CLASSES = {2:'car',3:'motorcycle',5:'bus',7:'truck'}
        COUNTING_LINES = {'default':((.05,.5),(.95,.5))}
        custom_config = TRACKER_SETTINGS
        RUN_LOG_DIR = str(output_root/'crossing_logs')
        TRACKER_CONFIG_PATH = str(output_root/'bytetrack.yaml')
        plan_path = output_root/'selected_pairs.csv'
        plan_bytes = plan_path.read_bytes() if previous_contract else plan.to_csv(index=False).encode()
        contract = {'model':YOLO_MODEL_NAME, 'tracker':custom_config, 'lines':COUNTING_LINES,
                    'processor_sha256':hashlib.sha256(PROCESSOR_CODE.encode()).hexdigest(),
                    'plan_sha256':hashlib.sha256(plan_bytes).hexdigest(),
                    'plan_semantic_sha256':plan_fingerprint(plan),
                    'sampling_limits':sampling_limits,
                    'label_csv_sha256':hashlib.sha256(label_path.read_bytes()).hexdigest(),
                    'ultralytics_version':importlib.metadata.version('ultralytics'), 'version':5}
        signature = contract_signature(contract)
        contract_path = output_root/'checkpoint_contract.json'
        checkpoint = output_root/'extracted_features.csv'
        if checkpoint.exists() and not contract_path.exists():
            raise RuntimeError('Checkpoint contract is missing. Use a new EXPERIMENT_NAME.')
        if previous_contract:
            changes = checkpoint_changes(previous_contract,contract)
            if changes:
                (output_root/'checkpoint_mismatch.json').write_text(json.dumps(changes,indent=2))
                print('Changed checkpoint fields:',', '.join(changes))
                for field,values in changes.items():
                    print(field,'| saved:',str(values['saved'])[:100],'| current:',str(values['current'])[:100])
                raise RuntimeError('Resume blocked; changed fields: '+', '.join(changes)+'. Existing checkpoints were not replaced. Details: '+str(output_root/'checkpoint_mismatch.json'))
            if previous_contract['configuration'].get('version') == 4:
                backup = output_root/'checkpoint_contract.v4.json'
                if not backup.exists():
                    backup.write_text(json.dumps(previous_contract,indent=2))
                print('Verified original plan/settings; upgraded resume metadata from v4 to v5.')
        Path(RUN_LOG_DIR).mkdir(exist_ok=True)
        Path(TRACKER_CONFIG_PATH).write_text(yaml.safe_dump(custom_config))
        pending_contract = contract_path.with_suffix('.pending.json')
        pending_contract.write_text(json.dumps({'signature':signature,'configuration':contract},indent=2))
        pending_contract.replace(contract_path)
        if not previous_contract:
            plan_path.write_bytes(plan_bytes)
        cached = pd.read_csv(checkpoint) if checkpoint.exists() else historical.copy()
        if not cached.model_name.eq(YOLO_MODEL_NAME).all():
            raise ValueError('Checkpoint model mismatch')
        cached = cached[cached.video_path.isin(plan.videos)].drop_duplicates('video_path', keep='last')
        rows = cached.to_dict('records')
        done = set(cached.video_path)
        missing = [key for key in plan.videos if key not in done]
        pending = checkpoint.with_suffix('.pending.csv')
        cached.to_csv(pending,index=False)
        pending.replace(checkpoint)
        print('Completed clips reused:',len(done),'| New clips to process:',len(missing),flush=True)

        def make_video_key(path):
            return Path(path).relative_to(drive_base).as_posix()

        exec(PROCESSOR_CODE, globals())
        if missing:
            model = YOLO(YOLO_MODEL_NAME)
            extraction_start = time.perf_counter()
            for index,key in enumerate(missing,1):
                print(f'[{index}/{len(missing)}] Processing {key}',flush=True)
                row,_,_ = process_video_with_tracking(available[key],model)
                if row is None:
                    raise RuntimeError('Interrupted. Run from the top with unchanged settings to resume completed clips.')
                if row['video_path'] != key or row['model_name'] != YOLO_MODEL_NAME:
                    raise ValueError('Extractor returned an unexpected video or model.')
                rows.append(row)
                pd.DataFrame(rows).to_csv(pending,index=False)
                pending.replace(checkpoint)
                elapsed = time.perf_counter()-extraction_start
                eta = elapsed/index*(len(missing)-index)/60
                print(f'Checkpoint saved: {len(rows)}/{len(plan)} clips; estimated remaining {eta:.1f} minutes at current speed.',flush=True)
        features = pd.DataFrame(rows)
        if features.video_path.duplicated().any() or set(features.video_path) != set(plan.videos):
            raise RuntimeError('Every selected clip must finish once before evaluation.')
        metadata = ['videos','source_time','target_time','source_date','split',
                    'next_enter_rating','congestion_enter_rating','horizon_sec',
                    'target_video_path','gap_sec']
        features = features.drop(columns=[c for c in metadata if c in features])
        features = features.merge(plan[metadata],left_on='video_path',right_on='videos',validate='one_to_one')
        if USE_MOTION_FEATURES:
            old_logs = drive_base/('crossing_logs_v2' if DETECTOR=='yolo11' else 'crossing_logs_yolov8_v2')
            features = add_motion(features,[RUN_LOG_DIR,old_logs])
        features.to_csv(output_root/'matched_features.csv',index=False)
else:
    features = historical.copy()
    RESULT_DIR = output_root/'cached_evaluation'
    EVALUATION_KIND = 'reused_historical_holdout'

In [ ]:
if RUN_MODE == 'expanded' and PLAN_ONLY:
    print('Preflight finished; no forecasting evaluation was run.')
else:
    selection, metrics = evaluate_pipeline(features, RESULT_DIR, EVALUATION_KIND)
    display(metrics)
    print('Class distribution (compare with majority and persistence on this same sample):')
    display(pd.read_csv(RESULT_DIR/'split_class_distribution.csv'))
    import matplotlib.pyplot as plt
    matrix = pd.read_csv(RESULT_DIR/'confusion_matrix.csv', index_col=0)
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(matrix.values, cmap='Blues')
    for i in range(4):
        for j in range(4):
            r,g,b,_ = im.cmap(im.norm(matrix.iloc[i,j]))
            color = 'black' if .2126*r+.7152*g+.0722*b>.5 else 'white'
            ax.text(j,i,str(matrix.iloc[i,j]),ha='center',va='center',color=color,fontsize=14)
    ax.set_xticks(range(4),CLASSES,rotation=25,ha='right')
    ax.set_yticks(range(4),CLASSES)
    ax.set_xlabel('Predicted future class'); ax.set_ylabel('True future class')
    ax.set_title(DETECTOR+' features: '+selection['selected_candidate'])
    fig.tight_layout(); fig.savefig(RESULT_DIR/'confusion_matrix.png',dpi=300); plt.show()
    import shutil
    if RUN_MODE == 'expanded':
        metadata_folder = RESULT_DIR/'run_metadata'
        metadata_folder.mkdir(exist_ok=True)
        for name in ['dataset_inventory.csv','labelled_videos_missing_from_drive.csv',
                     'videos_without_labels.csv','invalid_label_timing.csv',
                     'selected_pairs.csv','checkpoint_contract.json']:
            shutil.copy2(output_root/name,metadata_folder/name)
    archive = shutil.make_archive(str(RESULT_DIR.parent/('Forecasting_v3_'+DETECTOR+'_'+RUN_MODE)), 'zip', RESULT_DIR)
    print('Results archive:',archive)
    print('These recording dates were previously examined; independent confirmation on new dates is still required.')
    print('Google Drive results folder:',RESULT_DIR.parent)

## Saving and resuming

Results are already saved in the experiment folder in Google Drive. Download
the `Forecasting_v3_yolo11_expanded.zip` archive from there after completion.
The final cell can recreate that ZIP without retraining in the same runtime.

## If execution disconnects

Reconnect a Colab GPU, authorize the same Drive account, and run from the top.
Keep `EXPERIMENT_NAME`, detector, sample sizes, labels and settings unchanged.
Completed clips are read from `extracted_features.csv` and skipped.

In [ ]:
if RUN_MODE == 'expanded' and PLAN_ONLY:
    print('Preflight reports are already saved. No evaluation ZIP is needed yet.')
else:
    # Run only this cell to save an already completed run from this Colab runtime.
    from pathlib import Path
    from google.colab import drive
    import shutil
    import csv
    import json

    drive.mount('/content/drive')
    save_base = Path(globals().get('DRIVE_BASE', '/content/drive/MyDrive/CP-1 Dataset'))
    if not save_base.is_dir():
        raise FileNotFoundError('CP-1 Dataset is not accessible. Check the mounted account and My Drive shortcut.')
    save_detector = globals().get('DETECTOR', 'yolo11')
    save_mode = globals().get('RUN_MODE', 'cached')
    if save_detector not in {'yolo11', 'yolov8'} or save_mode not in {'cached', 'expanded'}:
        raise ValueError('Check DETECTOR and RUN_MODE before saving.')
    save_root = save_base/'Forecasting_v3'/save_detector
    if save_mode == 'expanded':
        if globals().get('PLAN_ONLY', False):
            raise RuntimeError('Preflight only: set PLAN_ONLY=False and run from the top before saving evaluation results.')
        save_root = save_root/globals().get('EXPERIMENT_NAME','expanded_01')
    evaluation_name = 'cached_evaluation' if save_mode == 'cached' else 'evaluation'
    saved_results = save_root/evaluation_name
    runtime_results = Path('/content/Forecasting_v3')/save_detector/evaluation_name
    current_results = Path(globals().get('RESULT_DIR', runtime_results))
    source_results = next((p for p in [current_results, runtime_results, saved_results]
                           if (p/'evaluation_metrics.csv').is_file()), None)
    if source_results is None:
        raise FileNotFoundError('Completed results are unavailable in this runtime. Rerun the evaluation/export cell.')
    with (source_results/'features_used.csv').open(newline='') as saved_table:
        saved_models = {row['model_name'] for row in csv.DictReader(saved_table)}
    expected_model = 'yolov8l.pt' if save_detector == 'yolov8' else 'yolo11l.pt'
    saved_selection = json.loads((source_results/'selected_model.json').read_text())
    expected_kind = 'reused_historical_holdout' if save_mode == 'cached' else 'new_clips_on_previously_examined_dates'
    if saved_models != {expected_model} or saved_selection['evaluation_kind'] != expected_kind:
        raise ValueError('Completed results do not match the selected detector/mode. Run the configuration and processing cells in order.')
    save_root.mkdir(parents=True, exist_ok=True)
    if source_results.resolve() != saved_results.resolve():
        shutil.copytree(source_results, saved_results, dirs_exist_ok=True)
    archive = shutil.make_archive(str(save_root/f'Forecasting_v3_{save_detector}_{save_mode}'),
                                  'zip', saved_results)
    RESULT_DIR = saved_results
    print('Results folder:', saved_results)
    print('Results archive:', archive)
    print('Open Google Drive > My Drive > CP-1 Dataset > Forecasting_v3 > '+save_detector)